# 🌄 Waterfall Valley: photoreal Blender Cycles render (free Colab GPU)

**How to use it (on your phone too):**
1. Top menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **in order, one at a time** (▶ button). Every cell shows its output.
3. **Cell 6 (preview)** renders a test frame. Check it; if there's an error, send a screenshot to Claude.
4. **Cell 7** renders all 480 frames. Frames are saved to Google Drive, so if Colab disconnects,
   run cells 1-3 again and then cell 7: it continues from where it stopped.
5. **Cell 8** builds the graded MP4 with sound and downloads it.

Time on a T4: roughly 60-120 s per frame, so about 8-16 hours in total, split across several Colab sessions.
To go faster, set `SAMPLES_FINAL = 32` in cell 4 (a bit more noise; the denoiser cleans most of it).

In [ ]:
# 1) GPU check
!nvidia-smi -L

In [ ]:
# 2) Storage: Google Drive (recommended, so a disconnect does not lose frames)
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/waterfall_valley'
else:
    OUT = '/content/waterfall_valley'
os.makedirs(OUT + '/frames', exist_ok=True)
print('Output folder:', OUT)

In [ ]:
# 3) Install Blender 4.2 LTS (headless) - about 2 minutes
import re, urllib.request, os
if not os.path.exists('/content/blender/blender'):
    html = urllib.request.urlopen('https://download.blender.org/release/Blender4.2/').read().decode()
    vers = sorted(set(re.findall(r'blender-(4\.2\.\d+)-linux-x64\.tar\.xz', html)), key=lambda v: int(v.split('.')[-1]))
    v = vers[-1]
    print('Downloading Blender', v)
    !wget -q https://download.blender.org/release/Blender4.2/blender-{v}-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1 && rm /content/blender.tar.xz
!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 libice6 > /dev/null 2>&1
!/content/blender/blender -b --version | head -1
os.makedirs('/content/wf', exist_ok=True)

## 4) Scripts (just run these cells; they only write files)

In [ ]:
%%writefile /content/wf/build_waterfall_scene.py
"""
WATERFALL VALLEY - Blender build script
=======================================
STEP 1 of 8: Plan + Clay match (geometry and camera only, grey clay override).

How to run (Blender 4.2+ / 5.x):
  A) Blender UI: File > New > General, then the Scripting tab > Open > this file > Run Script.
     WARNING: the script deletes every object in the CURRENT scene first. Use a new file.
  B) Command line (renders a clay preview still automatically):
     blender -b -P build_waterfall_scene.py

Coordinates: metres. +Y = camera looking forward (upstream), +X = right, +Z = up.
The layout is matched to the LEFT column of the reference sheet (Final/Clay left panel):
thin waterfall right of centre, layered cliffs left and right, river running to camera,
big smooth boulders in the foreground, pines on the slopes, peak in the centre gap.

All noise is pure Python (no mathutils), so the same geometry can be generated outside
Blender for offline checks. Building takes about 1-2 minutes.
"""

import math
import os
import random

try:
    import bpy
except ImportError:  # dry run outside Blender (offline preview / tests)
    bpy = None

# ---------------------------------------------------------------- CONFIG
MIRROR_X = False          # True = flip the whole layout left/right (waterfall left, sun top-left)
USE_CLAY_OVERRIDE = True  # Step 1: everything renders as matte grey clay
SEED = 7
PREVIEW_PERCENT = 50      # 50% of 1920x1080 for the clay test
PREVIEW_SAMPLES = 32
OUTPUT_DIR = os.path.join(os.path.expanduser("~"), "waterfall_project")
SAVE_BLEND = True
RENDER_STILL_IN_BACKGROUND = True   # only when run with blender -b

CAM_LOC = (0.0, -2.0, 1.8)
CAM_PITCH_DEG = 6.0       # tilt up from horizontal
CAM_LENS = 24.0
SUN_AZIMUTH_DEG = 24.0    # degrees to the right of the camera's forward direction
SUN_ELEVATION_DEG = 25.0

COLLECTIONS = ["Terrain", "Water", "Rocks", "Vegetation", "Leaves", "Birds", "Lights", "Camera"]


def mx(x):
    return -x if MIRROR_X else x


def P(x, y):
    return (mx(x), y)


# ---------------------------------------------------------------- MATH / NOISE
def clamp(v, a=0.0, b=1.0):
    return a if v < a else b if v > b else v


def lerp(a, b, t):
    return a + (b - a) * t


def smoothstep(e0, e1, x):
    t = clamp((x - e0) / (e1 - e0))
    return t * t * (3.0 - 2.0 * t)


def _hash3(ix, iy, iz, seed):
    h = (ix * 374761393 + iy * 668265263 + iz * 1274126177 + seed * 2246822519) & 0xFFFFFFFF
    h = ((h ^ (h >> 13)) * 3266489917) & 0xFFFFFFFF
    h ^= h >> 16
    return (h & 0xFFFF) / 32767.5 - 1.0


def hash1(i, seed):
    return (_hash3(int(i), 17, 31, seed) + 1.0) * 0.5


def vnoise(x, y, z=0.0, seed=0):
    """Smooth 3D value noise, range about -1..1."""
    ix, iy, iz = math.floor(x), math.floor(y), math.floor(z)
    fx, fy, fz = x - ix, y - iy, z - iz
    ux = fx * fx * fx * (fx * (fx * 6 - 15) + 10)
    uy = fy * fy * fy * (fy * (fy * 6 - 15) + 10)
    uz = fz * fz * fz * (fz * (fz * 6 - 15) + 10)
    h = _hash3
    x00 = lerp(h(ix, iy, iz, seed), h(ix + 1, iy, iz, seed), ux)
    x10 = lerp(h(ix, iy + 1, iz, seed), h(ix + 1, iy + 1, iz, seed), ux)
    x01 = lerp(h(ix, iy, iz + 1, seed), h(ix + 1, iy, iz + 1, seed), ux)
    x11 = lerp(h(ix, iy + 1, iz + 1, seed), h(ix + 1, iy + 1, iz + 1, seed), ux)
    return lerp(lerp(x00, x10, uy), lerp(x01, x11, uy), uz)


def fbm(x, y, z=0.0, octaves=4, seed=0, lac=2.03, gain=0.5):
    s, a, f, norm = 0.0, 1.0, 1.0, 0.0
    for i in range(octaves):
        s += a * vnoise(x * f + i * 17.31, y * f + i * 9.13, z * f + i * 3.7, seed + i)
        norm += a
        a *= gain
        f *= lac
    return s / norm


def catmull(p0, p1, p2, p3, t):
    t2, t3 = t * t, t * t * t
    return tuple(
        0.5 * ((2 * p1[k]) + (-p0[k] + p2[k]) * t + (2 * p0[k] - 5 * p1[k] + 4 * p2[k] - p3[k]) * t2
               + (-p0[k] + 3 * p1[k] - 3 * p2[k] + p3[k]) * t3)
        for k in range(2))


def icosphere(subdiv):
    t = (1.0 + 5 ** 0.5) / 2.0
    verts = [(-1, t, 0), (1, t, 0), (-1, -t, 0), (1, -t, 0), (0, -1, t), (0, 1, t),
             (0, -1, -t), (0, 1, -t), (t, 0, -1), (t, 0, 1), (-t, 0, -1), (-t, 0, 1)]
    verts = [tuple(c / math.sqrt(sum(q * q for q in v)) for c in v) for v in verts]
    faces = [(0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11), (1, 5, 9), (5, 11, 4),
             (11, 10, 2), (10, 7, 6), (7, 1, 8), (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8),
             (3, 8, 9), (4, 9, 5), (2, 4, 11), (6, 2, 10), (8, 6, 7), (9, 8, 1)]
    for _ in range(subdiv):
        cache, new_faces = {}, []

        def mid(a, b):
            key = (a, b) if a < b else (b, a)
            if key not in cache:
                va, vb = verts[a], verts[b]
                m = [(va[k] + vb[k]) / 2 for k in range(3)]
                ln = math.sqrt(sum(q * q for q in m))
                verts.append(tuple(q / ln for q in m))
                cache[key] = len(verts) - 1
            return cache[key]

        for a, b, c in faces:
            ab, bc, ca = mid(a, b), mid(b, c), mid(c, a)
            new_faces += [(a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca)]
        faces = new_faces
    return verts, faces


def grid_faces(cols, rows):
    """Quad faces for a (cols x rows) vertex grid stored column-major (index = i*rows + j)."""
    return [(i * rows + j, (i + 1) * rows + j, (i + 1) * rows + j + 1, i * rows + j + 1)
            for i in range(cols - 1) for j in range(rows - 1)]


# ---------------------------------------------------------------- RIVER + TERRAIN
RIVER_PTS = [P(40, 92), P(33, 75), P(22, 58), P(11, 40), P(4, 22), P(0, 5), P(0, -25)]
RIVER_HALF_W = [7.0, 7.5, 8.5, 9.5, 10.5, 12.0, 13.0]
POOL_C, POOL_R = P(40, 91), 11.0


def river_info(x, y):
    """Distance to river centreline and the river half-width at the closest point."""
    best_d, best_w = 1e9, RIVER_HALF_W[-1]
    for i in range(len(RIVER_PTS) - 1):
        (ax, ay), (bx, by) = RIVER_PTS[i], RIVER_PTS[i + 1]
        dx, dy = bx - ax, by - ay
        s = clamp(((x - ax) * dx + (y - ay) * dy) / (dx * dx + dy * dy))
        px, py = ax + dx * s, ay + dy * s
        d = math.hypot(x - px, y - py)
        if d < best_d:
            best_d, best_w = d, lerp(RIVER_HALF_W[i], RIVER_HALF_W[i + 1], s)
    return best_d, best_w


def terrain_h(x, y):
    d, rw = river_info(x, y)
    if d < rw:                                    # river channel (water surface is z=0)
        k = d / rw
        h = -0.15 - 0.85 * (1.0 - k * k)
    else:                                         # bank, then steep grassy slopes
        e = d - rw
        h = -0.15 + 1.6 * smoothstep(0.0, 6.0, e) + 0.35 * max(0.0, e - 8.0)
        h = min(h, 22.0)
        h += 2.2 * fbm(x * 0.035, y * 0.035, 0.0, 4, SEED + 1) * smoothstep(0.0, 8.0, e)
    h += 0.25 * fbm(x * 0.35, y * 0.35, 3.1, 3, SEED + 2)
    pd = math.hypot(x - POOL_C[0], y - POOL_C[1])  # plunge pool
    w = smoothstep(POOL_R, POOL_R * 0.55, pd)
    h = lerp(h, -1.7 * (1.0 - (pd / POOL_R) ** 2) - 0.2, w)
    return h


# ---------------------------------------------------------------- CLIFFS
class Cliff:
    """A layered rock wall built as a vertical sheet that follows a plan curve.
    Displacement is horizontal, along the curve normal that faces the camera."""

    def __init__(self, name, ctrl, heights, seed, base_z=-6.0):
        self.name, self.seed, self.base_z = name, seed, base_z
        self.recess = None   # (u_centre, width, depth)
        self.notch = None    # (u_centre, width, z_top)
        pts, hs = [], []
        n = len(ctrl)
        for i in range(n - 1):
            p0, p1, p2, p3 = ctrl[max(i - 1, 0)], ctrl[i], ctrl[i + 1], ctrl[min(i + 2, n - 1)]
            for k in range(40):
                t = k / 40.0
                pts.append(catmull(p0, p1, p2, p3, t))
                hs.append(lerp(heights[i], heights[i + 1], smoothstep(0.0, 1.0, t)))
        pts.append(ctrl[-1])
        hs.append(heights[-1])
        self.pts, self.hs = pts, hs
        self.cum = [0.0]
        for i in range(1, len(pts)):
            self.cum.append(self.cum[-1] + math.dist(pts[i - 1], pts[i]))
        self.length = self.cum[-1]

    def _idx(self, u):
        lo, hi = 0, len(self.cum) - 1
        u = clamp(u, 0.0, self.length)
        while hi - lo > 1:
            m = (lo + hi) // 2
            if self.cum[m] <= u:
                lo = m
            else:
                hi = m
        seg = self.cum[hi] - self.cum[lo]
        return lo, hi, (u - self.cum[lo]) / seg if seg > 0 else 0.0

    def frame(self, u):
        """Position (x, y) and camera-facing normal (nx, ny) at arc length u."""
        lo, hi, t = self._idx(u)
        a, b = self.pts[lo], self.pts[hi]
        px, py = lerp(a[0], b[0], t), lerp(a[1], b[1], t)
        tx, ty = b[0] - a[0], b[1] - a[1]
        ln = math.hypot(tx, ty) or 1.0
        tx, ty = tx / ln, ty / ln
        nx, ny = ty, -tx
        if nx * (CAM_LOC[0] - px) + ny * (CAM_LOC[1] - py) < 0:
            nx, ny = -nx, -ny
        return px, py, nx, ny, tx, ty

    def top(self, u):
        lo, hi, t = self._idx(u)
        h = lerp(self.hs[lo], self.hs[hi], t) + 4.0 * fbm(u * 0.04, 0.0, 0.0, 3, self.seed + 20)
        if self.notch:
            uc, w, zt = self.notch
            g = math.exp(-((u - uc) / w) ** 2)
            h = lerp(h, min(h, zt), g)
        return h

    def disp(self, u, z):
        s = self.seed
        top = self.top(u)
        zz = z + 0.04 * u + 2.0 * fbm(u * 0.015, 0.5, 0.0, 2, s + 11)
        lt = 3.4                                   # strata layer thickness
        li = math.floor(zz / lt)
        fr = zz / lt - li
        strata = (hash1(li, s) - 0.5) * 1.5 + 0.6 * smoothstep(0.0, 0.18, fr) - 0.6
        big = 6.0 * fbm(u * 0.025, z * 0.03, 0.3, 3, s + 1)
        mid = 1.6 * fbm(u * 0.11, z * 0.16, 1.7, 3, s + 2)
        fine = 0.35 * fbm(u * 0.55, z * 0.55, 2.9, 2, s + 3)
        crev = -2.2 * (1.0 - abs(vnoise(u * 0.16, 0.37, z * 0.012, s + 4))) ** 6
        roll = -6.0 * smoothstep(top - 7.0, top, z) ** 2
        talus = 3.5 * (1.0 - smoothstep(self.base_z, self.base_z + 9.0, z))
        d = big + mid + fine + strata + crev + roll + talus
        if self.recess:
            uc, w, depth = self.recess
            d -= depth * math.exp(-((u - uc) / w) ** 2)
        return d

    u_cut = None

    def mesh(self, du=0.7, dz=0.6):
        length = min(self.length, self.u_cut or self.length)
        cols = int(length / du) + 1
        max_top = max(self.top(i * du) for i in range(cols))
        rows = int((max_top - self.base_z) / dz) + 1
        verts = []
        for i in range(cols):
            u = min(i * du, length)
            px, py, nx, ny, _, _ = self.frame(u)
            tp = self.top(u)
            for j in range(rows):
                z = self.base_z + (tp - self.base_z) * j / (rows - 1)
                d = self.disp(u, z)
                verts.append((px + nx * d, py + ny * d, z))
        faces = grid_faces(cols, rows)
        # make face normals point toward the camera (solidify then grows backwards)
        px, py, nx, ny, tx, ty = self.frame(self.length * 0.5)
        if ty * nx - tx * ny < 0:
            faces = [tuple(reversed(f)) for f in faces]
        return verts, faces

    def u_nearest(self, x, y):
        best, bu = 1e18, 0.0
        for i, p in enumerate(self.pts):
            d = (p[0] - x) ** 2 + (p[1] - y) ** 2
            if d < best:
                best, bu = d, self.cum[i]
        return bu

    def signed_dist(self, x, y):
        """>0 = in front of the cliff base line (camera side)."""
        u = self.u_nearest(x, y)
        px, py, nx, ny, _, _ = self.frame(u)
        return (x - px) * nx + (y - py) * ny, u


WATERFALL_TOP = 27.0


def make_cliffs():
    """One amphitheatre wall: tall left mass -> lower centre face -> waterfall notch -> tall right wall.
    Split in two objects at the waterfall so each stays a manageable mesh."""
    left = Cliff("Cliff_Left_Centre",
                 [P(-64, 18), P(-50, 55), P(-38, 85), P(-18, 108), P(5, 115), P(25, 110), P(44, 100),
                  P(52, 86)],
                 [32.0, 45.0, 52.0, 48.0, 42.0, 29.0, 27.0, 40.0], SEED + 100)
    right = Cliff("Cliff_Right",
                  [P(47, 95), P(52, 86), P(60, 62), P(66, 36), P(70, 10)],
                  [30.0, 52.0, 55.0, 50.0, 42.0], SEED + 200)
    u_wf = left.u_nearest(*P(44, 100))
    left.recess = (u_wf, 7.0, 4.0)
    left.notch = (u_wf, 6.0, WATERFALL_TOP)
    left.u_cut = u_wf + 9.0            # left wall stops just past the falls, right wall takes over
    return left, right, u_wf


def waterfall_mesh(cliff, u_c, z_top, w_top, w_bot, out, seed, z_bot=-0.6, cols=14, dz=0.4):
    """Free-falling curtain: hugs the rock at the lip, never re-enters it, arcs outward."""
    rows = int((z_top - z_bot) / dz) + 1
    grid = [[None] * rows for _ in range(cols + 1)]
    for i in range(cols + 1):
        a = i / cols - 0.5
        d_prev = None
        for j in range(rows):
            z = z_top - (z_top - z_bot) * j / (rows - 1)
            f = (z_top - z) / (z_top - z_bot)
            w = lerp(w_top, w_bot, f ** 0.7)
            u = u_c + a * w
            px, py, nx, ny, _, _ = cliff.frame(u)
            rock = cliff.disp(u, z) + 0.45
            d = rock if d_prev is None else max(d_prev + 0.03, rock)
            d_prev = d
            dd = d + out * math.sqrt(f) + 0.3 * (1.0 - (2 * a) ** 2) \
                + 0.18 * fbm(a * 7.0, z * 0.06, 0.0, 2, seed)
            grid[i][j] = (px + nx * dd, py + ny * dd, z)
    verts = [grid[i][j] for i in range(cols + 1) for j in range(rows)]
    faces = grid_faces(cols + 1, rows)
    px, py, nx, ny, tx, ty = cliff.frame(u_c)
    if ty * nx - tx * ny < 0:
        faces = [tuple(reversed(f)) for f in faces]
    return verts, faces


# ---------------------------------------------------------------- PROPS (mesh data, unit size)
def boulder_mesh(seed, stretch=(1.3, 1.0, 0.75), subdiv=3):
    v, f = icosphere(subdiv)
    out = []
    for (x, y, z) in v:
        r = 1.0 + 0.14 * fbm(x * 1.3, y * 1.3, z * 1.3, 3, seed) + 0.03 * fbm(x * 4, y * 4, z * 4, 2, seed + 5)
        x, y, z = x * r, y * r, z * r
        if z < -0.35:                       # flat-ish bottom so it sits on the ground
            z = -0.35 + (z + 0.35) * 0.3
        out.append((x * stretch[0], y * stretch[1], z * stretch[2]))
    return out, f


def pine_mesh(seed, tiers=7, seg=9):
    """Height 1.0. Trunk + stacked drooping cones (clay blockout; Step 2B adds needle cards)."""
    rnd = random.Random(seed)
    verts, faces = [], []
    tr = 0.018                              # trunk
    base = len(verts)
    for k in range(6):
        a = k / 6 * math.tau
        verts += [(tr * math.cos(a), tr * math.sin(a), -0.05), (tr * math.cos(a), tr * math.sin(a), 0.35)]
    for k in range(6):
        k2 = (k + 1) % 6
        faces.append((base + 2 * k, base + 2 * k2, base + 2 * k2 + 1, base + 2 * k + 1))
    for t in range(tiers):
        f = t / (tiers - 1)
        zb = lerp(0.16, 0.80, f)
        hc = lerp(0.30, 0.22, f)
        r = lerp(0.26, 0.07, f) * rnd.uniform(0.9, 1.1)
        b0 = len(verts)
        for k in range(seg):
            a = k / seg * math.tau + rnd.uniform(-0.15, 0.15)
            rr = r * rnd.uniform(0.8, 1.15)
            verts.append((rr * math.cos(a), rr * math.sin(a), zb - rnd.uniform(0.0, 0.05)))
        verts.append((rnd.uniform(-0.01, 0.01), rnd.uniform(-0.01, 0.01), zb + hc))  # apex
        verts.append((0.0, 0.0, zb + 0.02))                                          # under-centre
        apex, cen = b0 + seg, b0 + seg + 1
        for k in range(seg):
            k2 = (k + 1) % seg
            faces.append((b0 + k, b0 + k2, apex))
            faces.append((b0 + k2, b0 + k, cen))
    return verts, faces


def bush_mesh(seed):
    v, f = icosphere(2)
    out = []
    for (x, y, z) in v:
        r = 1.0 + 0.3 * fbm(x * 2.2, y * 2.2, z * 2.2, 3, seed)
        out.append((x * r, y * r, max(z * r * 0.65, -0.15)))
    return out, f


# ---------------------------------------------------------------- MOUNTAINS
PEAKS = [(P(172, 520), 170.0, 90.0), (P(45, 620), 235.0, 110.0), (P(-200, 600), 200.0, 140.0),
         (P(400, 700), 215.0, 160.0), (P(-450, 750), 225.0, 180.0)]


def mountains_mesh(step=8.0):
    xs = [-900 + i * step for i in range(int(1800 / step) + 1)]
    ys = [280 + j * step for j in range(int(720 / step) + 1)]
    verts = []
    for x in xs:
        for y in ys:
            h = -8.0
            for (px, py), H, W in PEAKS:
                r2 = ((x - px) / W) ** 2 + ((y - py) / W) ** 2
                h = max(h, H * math.exp(-(r2 ** 0.8)))
            ridge = (1.0 - abs(vnoise(x * 0.012, y * 0.012, 0.0, SEED + 50))) ** 2
            h += (40.0 * ridge + 16.0 * fbm(x * 0.03, y * 0.03, 0.0, 4, SEED + 51)) * smoothstep(0.0, 60.0, h)
            verts.append((x, y, h))
    return verts, grid_faces(len(xs), len(ys))


# ---------------------------------------------------------------- BUILDER (bpy or dry-run)
class Builder:
    def __init__(self):
        self.records = []        # dry-run: (name, world_verts, faces, collection)
        self.colls = {}
        self.dry = bpy is None

    def reset(self):
        if self.dry:
            return
        for o in list(bpy.data.objects):
            bpy.data.objects.remove(o, do_unlink=True)
        for c in list(bpy.data.collections):
            bpy.data.collections.remove(c)
        for blocks in (bpy.data.meshes, bpy.data.materials, bpy.data.cameras, bpy.data.lights,
                       bpy.data.textures, bpy.data.node_groups, bpy.data.curves):
            for b in list(blocks):
                if b.users == 0:
                    blocks.remove(b)

    def collections(self, names):
        for n in names:
            if self.dry:
                self.colls[n] = n
            else:
                c = bpy.data.collections.new(n)
                bpy.context.scene.collection.children.link(c)
                self.colls[n] = c
        return self.colls

    def mesh(self, name, verts, faces, smooth=True):
        if self.dry:
            return (verts, faces)
        me = bpy.data.meshes.new(name)
        me.from_pydata(verts, [], faces)
        me.update()
        if smooth:
            try:
                me.shade_smooth()
            except AttributeError:
                me.polygons.foreach_set("use_smooth", [True] * len(me.polygons))
        return me

    def obj(self, name, mesh, coll, loc=(0, 0, 0), rot_z=0.0, scale=(1, 1, 1)):
        if self.dry:
            verts, faces = mesh
            c, s = math.cos(rot_z), math.sin(rot_z)
            wv = []
            for (x, y, z) in verts:
                x, y, z = x * scale[0], y * scale[1], z * scale[2]
                wv.append((x * c - y * s + loc[0], x * s + y * c + loc[1], z + loc[2]))
            self.records.append((name, wv, faces, coll))
            return None
        o = bpy.data.objects.new(name, mesh)
        self.colls[coll].objects.link(o)
        o.location = loc
        o.rotation_euler = (0.0, 0.0, rot_z)
        o.scale = scale
        return o


# ---------------------------------------------------------------- BUILD STEPS
def build_terrain(B, step=1.0):
    xs = [-90 + i * step for i in range(int(180 / step) + 1)]
    ys = [-30 + j * step for j in range(int(200 / step) + 1)]
    verts = [(x, y, terrain_h(x, y)) for x in xs for y in ys]
    B.obj("Terrain_Valley", B.mesh("Terrain_Valley", verts, grid_faces(len(xs), len(ys))), "Terrain")
    B.obj("Terrain_Mountains", B.mesh("Terrain_Mountains", *mountains_mesh()), "Terrain")


def build_cliffs(B, left, right, u_wf):
    for cl in (left, right):
        o = B.obj(cl.name, B.mesh(cl.name, *cl.mesh()), "Rocks")
        if o is not None:
            m = o.modifiers.new("Solidify", "SOLIDIFY")
            m.thickness = 12.0
            m.offset = -1.0
    B.obj("Waterfall_Main", B.mesh("Waterfall_Main",
          *waterfall_mesh(left, u_wf, WATERFALL_TOP, 10.0, 14.0, 2.5, SEED + 300, cols=24)), "Water")
    B.obj("Waterfall_Strand", B.mesh("Waterfall_Strand",
          *waterfall_mesh(left, u_wf - 7.5, 23.0, 1.0, 1.8, 1.4, SEED + 301, cols=6)), "Water")


def build_water(B):
    verts = [(x, y, 0.0) for x in (-45.0, 55.0) for y in (-30.0, 105.0)]
    B.obj("River_Surface", B.mesh("River_Surface", verts, [(0, 2, 3, 1)], smooth=False), "Water")


def build_rocks(B, rnd):
    hero = [  # (x, y, size, seed) -- big smooth boulders matched to the clay frame
        (-4.0, 21.0, 2.1, 1), (-7.5, 27.0, 2.0, 2), (8.0, 13.0, 1.8, 3), (-11.0, 16.0, 1.3, 4),
        (-14.0, 31.0, 1.6, 5), (5.0, 33.0, 1.0, 6),
        (14.0, 42.0, 3.3, 7), (20.0, 46.0, 2.8, 8), (26.0, 50.0, 3.8, 9), (9.0, 38.0, 2.2, 10),
        (31.0, 55.0, 3.0, 11), (36.0, 60.0, 3.5, 12), (17.0, 52.0, 2.1, 13), (42.0, 68.0, 3.6, 14),
        (25.0, 40.0, 2.0, 15), (3.0, 45.0, 1.8, 16), (34.0, 82.0, 3.0, 17), (48.0, 84.0, 3.2, 18),
    ]
    for i, (x, y, s, sd) in enumerate(hero):
        x = mx(x)
        st = (rnd.uniform(1.1, 1.5), rnd.uniform(0.9, 1.1), rnd.uniform(0.65, 0.85))
        me = B.mesh(f"Boulder_Hero_{i:02d}", *boulder_mesh(SEED + 400 + sd, st))
        B.obj(f"Boulder_Hero_{i:02d}", me, "Rocks", (x, y, terrain_h(x, y) - 0.2 * s),
              rnd.uniform(0, math.tau), (s, s, s))
    variants = [B.mesh(f"Boulder_Var_{k}", *boulder_mesh(SEED + 500 + k, (1.3, 1.0, 0.75 + 0.05 * k)))
                for k in range(6)]
    pebbles = [B.mesh(f"Pebble_Var_{k}", *boulder_mesh(SEED + 600 + k, (1.4, 1.0, 0.6), subdiv=1))
               for k in range(5)]
    n = 0
    for _ in range(400):                      # bank boulders along the river edges
        if n >= 30:
            break
        i = rnd.randrange(len(RIVER_PTS) - 1)
        (ax, ay), (bx, by) = RIVER_PTS[i], RIVER_PTS[i + 1]
        t = rnd.random()
        cx, cy = lerp(ax, bx, t), lerp(ay, by, t)
        if cy < 25:
            continue
        tx, ty = bx - ax, by - ay
        ln = math.hypot(tx, ty)
        nx, ny = -ty / ln, tx / ln
        side = rnd.choice((-1, 1))
        rw = lerp(RIVER_HALF_W[i], RIVER_HALF_W[i + 1], t)
        off = rw + rnd.uniform(-2.5, 3.0)
        x, y = cx + nx * off * side, cy + ny * off * side
        s = rnd.uniform(0.4, 1.5) * (1.2 if cy < 40 else 0.9)
        B.obj(f"Boulder_{n:03d}", rnd.choice(variants), "Rocks",
              (x, y, terrain_h(x, y) - 0.25 * s), rnd.uniform(0, math.tau), (s, s, s))
        n += 1
    for k in range(90):                        # riverbed pebbles / stones breaking the surface
        i = rnd.randrange(len(RIVER_PTS) - 1)
        (ax, ay), (bx, by) = RIVER_PTS[i], RIVER_PTS[i + 1]
        t = rnd.random()
        rw = lerp(RIVER_HALF_W[i], RIVER_HALF_W[i + 1], t)
        tx, ty = bx - ax, by - ay
        ln = math.hypot(tx, ty)
        off = rnd.uniform(-rw, rw)
        x, y = lerp(ax, bx, t) - ty / ln * off, lerp(ay, by, t) + tx / ln * off
        if y < 30:
            continue
        s = rnd.uniform(0.25, 0.7)
        B.obj(f"Pebble_{k:03d}", rnd.choice(pebbles), "Rocks",
              (x, y, max(terrain_h(x, y), -0.55) + 0.05), rnd.uniform(0, math.tau), (s, s, s))


def build_vegetation(B, rnd, left, right):
    pines = [B.mesh(f"Pine_Var_{k}", *pine_mesh(SEED + 700 + k), smooth=False) for k in range(4)]
    bushes = [B.mesh(f"Bush_Var_{k}", *bush_mesh(SEED + 800 + k)) for k in range(4)]

    def free_ground(x, y, margin):
        d, rw = river_info(x, y)
        if d < rw + margin:
            return False
        for cl in (left, right):
            sd, u = cl.signed_dist(x, y)
            if 0 < u < cl.length and sd < 9.0:
                return False
        return True

    def place_pine(name, x, y, z, h):
        B.obj(name, rnd.choice(pines), "Vegetation", (x, y, z - 0.3), rnd.uniform(0, math.tau),
              (h * rnd.uniform(0.9, 1.1), h * rnd.uniform(0.9, 1.1), h))

    n = 0
    for _ in range(3000):                      # conifer cluster on the left slope
        if n >= 20:
            break
        x, y = rnd.uniform(-55, -18), rnd.uniform(50, 88)
        x = mx(x)
        if not free_ground(x, y, 5.0):
            continue
        place_pine(f"Pine_Slope_{n:03d}", x, y, terrain_h(x, y), rnd.uniform(14, 22))
        n += 1
    hx, hy = P(-5, 52)                         # the single tall pine left of centre
    place_pine("Pine_Hero", hx, hy, terrain_h(hx, hy), 22.0)
    n = 0
    u_max = left.u_nearest(*P(20, 112))        # cliff-top trees only on the left / centre wall
    for _ in range(14):
        u = rnd.uniform(4, u_max)
        px, py, nx, ny, _, _ = left.frame(u)
        back = rnd.uniform(-11.0, -8.0) + left.disp(u, left.top(u))
        place_pine(f"Pine_Top_{n:03d}", px + nx * back, py + ny * back, left.top(u) + 0.2,
                   rnd.uniform(9, 15))
        n += 1
    n = 0
    for _ in range(3000):                      # bushes on the banks
        if n >= 45:
            break
        x, y = rnd.uniform(-35, 15), rnd.uniform(8, 75)
        x = mx(x)
        d, rw = river_info(x, y)
        if not (rw + 0.5 < d < rw + 9.0) or not free_ground(x, y, 0.5):
            continue
        s = rnd.uniform(0.5, 1.6)
        B.obj(f"Bush_{n:03d}", rnd.choice(bushes), "Vegetation", (x, y, terrain_h(x, y) - 0.1),
              rnd.uniform(0, math.tau), (s * 1.2, s, s))
        n += 1


# ---------------------------------------------------------------- BLENDER-ONLY SETUP
def sun_direction():
    az, el = math.radians(mx(SUN_AZIMUTH_DEG)), math.radians(SUN_ELEVATION_DEG)
    return (math.sin(az) * math.cos(el), math.cos(az) * math.cos(el), math.sin(el))


def setup_camera_light_world(B):
    from mathutils import Vector
    scene = bpy.context.scene
    cam_data = bpy.data.cameras.new("CAM_Main")
    cam_data.lens = CAM_LENS
    cam_data.clip_start = 0.1
    cam_data.clip_end = 3000.0
    cam = bpy.data.objects.new("CAM_Main", cam_data)
    B.colls["Camera"].objects.link(cam)
    cam.location = CAM_LOC
    cam.rotation_euler = (math.radians(90.0 + CAM_PITCH_DEG), 0.0, 0.0)
    scene.camera = cam

    sun_data = bpy.data.lights.new("LGT_Sun_Key", "SUN")
    sun_data.energy = 4.0
    sun_data.angle = math.radians(1.5)
    sun = bpy.data.objects.new("LGT_Sun_Key", sun_data)
    B.colls["Lights"].objects.link(sun)
    s = Vector(sun_direction())
    sun.rotation_euler = (-s).to_track_quat("-Z", "Y").to_euler()
    sun.location = (s * 60.0)

    world = bpy.data.worlds.new("WLD_Clay")
    try:
        world.use_nodes = True
    except Exception:
        pass
    nt = world.node_tree
    bg = next((n for n in nt.nodes if n.type == "BACKGROUND"), None)
    if bg is None:
        bg = nt.nodes.new("ShaderNodeBackground")
        out = next((n for n in nt.nodes if n.type == "OUTPUT_WORLD"), None) or nt.nodes.new("ShaderNodeOutputWorld")
        nt.links.new(bg.outputs[0], out.inputs[0])
    bg.inputs["Color"].default_value = (0.62, 0.68, 0.76, 1.0)
    bg.inputs["Strength"].default_value = 1.0
    scene.world = world


def make_clay_material():
    mat = bpy.data.materials.new("MAT_Clay")
    try:
        mat.use_nodes = True
    except Exception:
        pass
    bsdf = next(n for n in mat.node_tree.nodes if n.type == "BSDF_PRINCIPLED")
    bsdf.inputs["Base Color"].default_value = (0.6, 0.6, 0.6, 1.0)
    bsdf.inputs["Roughness"].default_value = 0.8
    return mat


def setup_render():
    scene = bpy.context.scene
    r = scene.render
    r.engine = "CYCLES"
    r.resolution_x, r.resolution_y = 1920, 1080
    r.resolution_percentage = PREVIEW_PERCENT
    r.fps = 24
    scene.frame_start, scene.frame_end = 1, 480
    scene.frame_set(1)
    scene.cycles.samples = PREVIEW_SAMPLES
    scene.cycles.use_denoising = True
    scene.cycles.max_bounces = 6
    try:
        scene.view_settings.view_transform = "AgX"
    except TypeError:
        pass


def frame_viewport_on_camera():
    try:
        for area in bpy.context.screen.areas:
            if area.type == "VIEW_3D":
                area.spaces[0].region_3d.view_perspective = "CAMERA"
                area.spaces[0].shading.type = "SOLID"
    except Exception:
        pass


# ---------------------------------------------------------------- MAIN
def main():
    rnd = random.Random(SEED)
    B = Builder()
    B.reset()
    B.collections(COLLECTIONS)
    left, right, u_wf = make_cliffs()
    build_terrain(B)
    build_cliffs(B, left, right, u_wf)
    build_water(B)
    build_rocks(B, rnd)
    build_vegetation(B, rnd, left, right)
    if B.dry:
        return B
    setup_camera_light_world(B)
    setup_render()
    if USE_CLAY_OVERRIDE:
        bpy.context.view_layer.material_override = make_clay_material()
    frame_viewport_on_camera()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if SAVE_BLEND:
        bpy.ops.wm.save_as_mainfile(filepath=os.path.join(OUTPUT_DIR, "waterfall_valley.blend"))
    if bpy.app.background and RENDER_STILL_IN_BACKGROUND:
        bpy.context.scene.render.filepath = os.path.join(OUTPUT_DIR, "step1_clay.png")
        bpy.ops.render.render(write_still=True)
    print("Waterfall Valley step 1 built:", len(bpy.data.objects), "objects")
    return B


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/wf/final_scene.py
"""
WATERFALL VALLEY - photoreal Cycles build (Steps 2-5 on top of the Step 1 layout).

  Materials : procedural PBR (displacement + bump + roughness variation), strata cliffs, moss, golden
              lichen, dirt, wet rock near water, glacial turquoise water (transmission + volume), foam.
  Vegetation: weeping foreground tree with ~100k real leaf meshes (curved, folded, crumpled, 0.3 mm thick,
              size +-30 %, per-leaf colour, veins, translucency + SSS, waxy coat, spots, dry edges,
              insect holes), needle-cluster conifers, leafy bushes, Geometry Nodes grass (random scale
              0.8-1.2 and rotation), tall wild grass at the river edge.
  Light     : Nishita sky + clouds, warm sun lamp with soft shadows, path-traced GI, volumetric haze
              (god rays through the leaves), waterfall mist volume.
  Animation : 480 frames @ 24 fps - eased camera glide, 7 birds with flapping wings, looping wind on leaves,
              bushes and grass, flowing water and falling waterfall.
  Camera    : 36 mm full-frame sensor, shallow-ish DOF, chromatic aberration + glare in the compositor
              (grain, vignette and the final grade are added by ffmpeg when encoding).

Run (Blender 4.2 LTS; see the Colab notebook):
  blender -b -P final_scene.py -- --save waterfall_final.blend [--still 1 --out preview.png] [--preview]
"""
import math
import os
import random
import sys

import bpy
from mathutils import Quaternion, Vector

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import build_waterfall_scene as wf  # noqa: E402

# ------------------------------------------------------------------ CONFIG
SAMPLES_FINAL = 64
SAMPLES_PREVIEW = 24
SUN_AZIMUTH = 48.0        # degrees right of the camera axis (just outside the frame, top-right)
SUN_ELEVATION = 30.0
SUN_STRENGTH = 4.5
SKY_STRENGTH = 0.30
EXPOSURE = 0.0
HAZE_DENSITY = 0.0012
FSTOP = 2.2
GRASS_DENSITY = 40.0      # tufts per m2 at full weight
LEAF_STRANDS_PER_BRANCH = 46
MOTION_BLUR = True
FPS, FRAMES = 24, 480

ARGS = sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []


def arg(name, default=None, cast=str):
    if name in ARGS:
        i = ARGS.index(name)
        if i + 1 < len(ARGS) and not ARGS[i + 1].startswith("--"):
            return cast(ARGS[i + 1])
        return True
    return default


def lin(h):
    """'#RRGGBB' (sRGB) -> linear RGBA."""
    c = [int(h[i:i + 2], 16) / 255.0 for i in (1, 3, 5)]
    c = [x / 12.92 if x <= 0.04045 else ((x + 0.055) / 1.055) ** 2.4 for x in c]
    return (*c, 1.0)


def mx(x):
    return wf.mx(x)


# ------------------------------------------------------------------ NODE HELPERS
class NB:
    """Tiny node-tree builder for shader and geometry-node trees."""

    def __init__(self, nt):
        self.nt = nt

    def n(self, t, **kw):
        node = self.nt.nodes.new(t)
        for k, v in kw.items():
            try:
                setattr(node, k, v)
            except (AttributeError, TypeError) as e:
                print("WARN", t, k, e)
        return node

    def link(self, a, b):
        self.nt.links.new(a, b)

    def put(self, sock, v):
        if isinstance(v, bpy.types.NodeSocket):
            self.link(v, sock)
        elif v is not None:
            sock.default_value = v

    def setin(self, node, names, v):
        for nm in (names if isinstance(names, (list, tuple)) else (names,)):
            s = node.inputs.get(nm)
            if s is not None:
                self.put(s, v)
                return s
        print("WARN missing input", node.bl_idname, names)

    def math(self, op, a, b=None, c=None, clamp=False):
        node = self.n("ShaderNodeMath", operation=op, use_clamp=clamp)
        self.put(node.inputs[0], a)
        if b is not None:
            self.put(node.inputs[1], b)
        if c is not None:
            self.put(node.inputs[2], c)
        return node.outputs[0]

    def vmath(self, op, a, b=None):
        node = self.n("ShaderNodeVectorMath", operation=op)
        self.put(node.inputs[0], a)
        if b is not None:
            self.put(node.inputs[1], b)
        return node.outputs["Value"] if op in ("DOT_PRODUCT", "DISTANCE", "LENGTH") else node.outputs["Vector"]

    def xyz(self, x=0.0, y=0.0, z=0.0):
        node = self.n("ShaderNodeCombineXYZ")
        for i, v in enumerate((x, y, z)):
            self.put(node.inputs[i], v)
        return node.outputs[0]

    def sep(self, v):
        node = self.n("ShaderNodeSeparateXYZ")
        self.put(node.inputs[0], v)
        return node.outputs

    def ramp(self, fac, stops):
        node = self.n("ShaderNodeValToRGB")
        self.put(node.inputs["Fac"], fac)
        els = node.color_ramp.elements
        while len(els) < len(stops):
            els.new(0.5)
        for el, (pos, col) in zip(els, stops):
            el.position = pos
            el.color = col if len(col) == 4 else (*col, 1.0)
        return node.outputs["Color"]

    def ramp_f(self, fac, a, b):
        """Smooth 0..1 step of fac between a and b (as a float)."""
        return self._maprange(fac, a, b)

    def _maprange(self, fac, a, b):
        node = self.n("ShaderNodeMapRange", interpolation_type="SMOOTHSTEP", clamp=True)
        self.put(node.inputs["Value"], fac)
        node.inputs["From Min"].default_value = a
        node.inputs["From Max"].default_value = b
        return node.outputs["Result"]

    def mix(self, fac, a, b, blend="MIX"):
        node = self.n("ShaderNodeMixRGB", blend_type=blend)
        self.put(node.inputs["Fac"], fac)
        self.put(node.inputs["Color1"], a)
        self.put(node.inputs["Color2"], b)
        return node.outputs["Color"]

    def noise(self, vec, scale, detail=4.0, rough=0.55, dist=0.0, w=None):
        node = self.n("ShaderNodeTexNoise")
        if w is not None:
            node.noise_dimensions = "4D"
            self.put(node.inputs["W"], w)
        self.put(node.inputs["Vector"], vec)
        node.inputs["Scale"].default_value = scale
        node.inputs["Detail"].default_value = detail
        node.inputs["Roughness"].default_value = rough
        node.inputs["Distortion"].default_value = dist
        return node

    def voronoi(self, vec, scale, feature="F1", rand=1.0):
        node = self.n("ShaderNodeTexVoronoi", feature=feature)
        self.put(node.inputs["Vector"], vec)
        node.inputs["Scale"].default_value = scale
        node.inputs["Randomness"].default_value = rand
        return node

    def driver(self, sock, expr):
        fc = sock.driver_add("default_value")
        fc.driver.type = "SCRIPTED"
        fc.driver.expression = expr
        return fc


def new_material(name):
    mat = bpy.data.materials.new(name)
    try:
        mat.use_nodes = True
    except Exception:
        pass
    mat.node_tree.nodes.clear()
    b = NB(mat.node_tree)
    out = b.n("ShaderNodeOutputMaterial")
    return mat, b, out


def principled(b, **inputs):
    p = b.n("ShaderNodeBsdfPrincipled")
    names = {
        "base": ("Base Color",), "rough": ("Roughness",), "metal": ("Metallic",), "ior": ("IOR",),
        "alpha": ("Alpha",), "normal": ("Normal",), "spec": ("Specular IOR Level", "Specular"),
        "sss": ("Subsurface Weight", "Subsurface"), "sss_radius": ("Subsurface Radius",),
        "sss_scale": ("Subsurface Scale",), "trans": ("Transmission Weight", "Transmission"),
        "coat": ("Coat Weight", "Clearcoat"), "coat_rough": ("Coat Roughness", "Clearcoat Roughness"),
        "sheen": ("Sheen Weight", "Sheen"),
    }
    for k, v in inputs.items():
        b.setin(p, names[k], v)
    return p


def set_displacement(mat):
    for target in (mat, getattr(mat, "cycles", None)):
        try:
            target.displacement_method = "BOTH"
            return
        except Exception:
            pass


def assign(obj, mat):
    obj.data.materials.clear()
    obj.data.materials.append(mat)


def subsurf(obj, render_levels, simple=True):
    m = obj.modifiers.new("Subdiv", "SUBSURF")
    m.subdivision_type = "SIMPLE" if simple else "CATMULL_CLARK"
    m.levels = 0
    m.render_levels = render_levels
    return m


def link_obj(obj, coll):
    coll.objects.link(obj)
    return obj


def mesh_obj(name, verts, faces, coll, uvs=None, smooth=True):
    me = bpy.data.meshes.new(name)
    me.from_pydata(verts, [], faces)
    me.update()
    if uvs is not None:
        uvl = me.uv_layers.new(name="UVMap")
        for loop in me.loops:
            uvl.data[loop.index].uv = uvs[loop.vertex_index]
    if smooth and faces:
        try:
            me.shade_smooth()
        except AttributeError:
            me.polygons.foreach_set("use_smooth", [True] * len(me.polygons))
    obj = bpy.data.objects.new(name, me)
    if coll is not None:
        coll.objects.link(obj)
    return obj


def point_cloud_obj(name, pts, attrs, coll):
    """Vertex-only mesh with named point attributes (for Geometry Nodes instancing)."""
    me = bpy.data.meshes.new(name)
    me.from_pydata(pts, [], [])
    for aname, (atype, values) in attrs.items():
        a = me.attributes.new(aname, atype, "POINT")
        flat = []
        for v in values:
            flat.extend(v) if isinstance(v, (tuple, list, Vector)) else flat.append(v)
        a.data.foreach_set("vector" if atype == "FLOAT_VECTOR" else "value", flat)
    me.update()
    obj = bpy.data.objects.new(name, me)
    coll.objects.link(obj)
    return obj


def collection(name, parent=None):
    c = bpy.data.collections.new(name)
    (parent.children if parent else bpy.context.scene.collection.children).link(c)
    return c


# ------------------------------------------------------------------ MATERIALS
def mat_terrain():
    mat, b, out = new_material("MAT_Ground")
    tc = b.n("ShaderNodeTexCoord").outputs["Object"]
    e = b.n("ShaderNodeAttribute", attribute_name="e", attribute_type="GEOMETRY").outputs["Fac"]
    geo = b.n("ShaderNodeNewGeometry")
    nz = b.sep(geo.outputs["Normal"])[2]
    big = b.noise(tc, 0.15, 5, 0.6).outputs["Fac"]
    fine = b.noise(tc, 6.0, 6, 0.6).outputs["Fac"]
    stones = b.voronoi(tc, 2.2)
    pebbles = b.voronoi(tc, 9.0)
    # soil / grass floor under the grass blades
    soil = b.mix(big, lin("#4a5a22"), lin("#6b6a33"))
    soil = b.mix(b.math("MULTIPLY", fine, 0.6), soil, lin("#3b3423"))
    # steep earth
    earth = b.mix(fine, lin("#6e6250"), lin("#8a7c66"))
    steep = b.ramp_f(nz, 0.86, 0.72)
    col = b.mix(steep, soil, earth)
    # gravel band at the river edge, wet and dark close to the waterline
    gravel = b.mix(pebbles.outputs["Distance"], lin("#8f897d"), lin("#5f5a52"))
    gravel = b.mix(b.math("MULTIPLY", b.noise(tc, 20.0, 3).outputs["Fac"], 0.5), gravel, lin("#b2ab9c"))
    gmask = b.ramp_f(e, 3.0, 1.2)
    col = b.mix(gmask, col, gravel)
    bed = b.mix(stones.outputs["Distance"], lin("#5b5141"), lin("#3a3f2e"))
    bed = b.mix(b.math("MULTIPLY", big, 0.7), bed, lin("#2d4a3a"))         # algae
    bmask = b.ramp_f(e, 0.0, -0.6)
    col = b.mix(bmask, col, bed)
    wet = b.math("MULTIPLY", b.ramp_f(e, 1.4, 0.2), 1.0)
    col = b.mix(b.math("MULTIPLY", wet, 0.45), col, (0.0, 0.0, 0.0, 1.0))
    rough = b.math("SUBTRACT", b.math("ADD", 0.82, b.math("MULTIPLY", fine, 0.15)), b.math("MULTIPLY", wet, 0.65))
    height = b.math("ADD", b.math("MULTIPLY", fine, 0.5),
                    b.math("MULTIPLY", b.math("ADD", gmask, bmask), b.math("SUBTRACT", 1.0, pebbles.outputs["Distance"])))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.6
    bump.inputs["Distance"].default_value = 0.04
    b.link(height, bump.inputs["Height"])
    p = principled(b, base=col, rough=rough, normal=bump.outputs["Normal"], spec=0.4)
    disp = b.n("ShaderNodeDisplacement")
    disp.inputs["Scale"].default_value = 0.08
    disp.inputs["Midlevel"].default_value = 0.4
    b.link(height, disp.inputs["Height"])
    b.link(p.outputs[0], out.inputs["Surface"])
    b.link(disp.outputs[0], out.inputs["Displacement"])
    set_displacement(mat)
    return mat


def mat_cliff(fall_xy):
    mat, b, out = new_material("MAT_Cliff_Strata")
    pos = b.n("ShaderNodeTexCoord").outputs["Object"]
    geo = b.n("ShaderNodeNewGeometry")
    nz = b.sep(geo.outputs["Normal"])[2]
    wave = b.n("ShaderNodeTexWave", wave_type="BANDS", bands_direction="Z")
    b.link(pos, wave.inputs["Vector"])
    wave.inputs["Scale"].default_value = 0.045
    wave.inputs["Distortion"].default_value = 7.0
    wave.inputs["Detail"].default_value = 4.0
    strata = wave.outputs["Fac"]
    col = b.ramp(strata, [(0.0, lin("#8d8273")), (0.3, lin("#b5a58a")), (0.55, lin("#9a8d7a")),
                          (0.75, lin("#c2ad86")), (1.0, lin("#7a7268"))])
    big = b.noise(pos, 0.08, 5, 0.6).outputs["Fac"]
    fine = b.noise(pos, 3.0, 8, 0.65).outputs["Fac"]
    col = b.mix(b.math("MULTIPLY", big, 0.5), col, lin("#6d665c"), "MULTIPLY")
    cracks = b.voronoi(pos, 0.35, "DISTANCE_TO_EDGE").outputs["Distance"]
    crack_m = b.ramp_f(cracks, 0.06, 0.0)
    col = b.mix(b.math("MULTIPLY", crack_m, 0.8), col, lin("#2e2a25"))
    # vertical dirt streaks
    smap = b.n("ShaderNodeMapping")
    b.link(pos, smap.inputs["Vector"])
    smap.inputs["Scale"].default_value = (1.0, 1.0, 0.08)
    streak = b.ramp_f(b.noise(smap.outputs[0], 0.6, 4).outputs["Fac"], 0.5, 0.75)
    col = b.mix(b.math("MULTIPLY", streak, 0.45), col, lin("#4a4338"))
    # golden lichen and moss on ledges
    lich = b.ramp_f(b.noise(pos, 0.9, 6, 0.7).outputs["Fac"], 0.6, 0.68)
    col = b.mix(b.math("MULTIPLY", lich, 0.75), col, lin("#c49a3c"))
    moss_m = b.math("MULTIPLY", b.ramp_f(nz, 0.35, 0.75), b.ramp_f(big, 0.35, 0.55))
    moss = b.mix(fine, lin("#3e5a1c"), lin("#5f7a28"))
    col = b.mix(moss_m, col, moss)
    # wet, dark rock in the waterfall spray
    d = b.vmath("DISTANCE", pos, (fall_xy[0], fall_xy[1], 12.0))
    wet = b.ramp_f(d, 22.0, 8.0)
    col = b.mix(b.math("MULTIPLY", wet, 0.55), col, (0.0, 0.0, 0.0, 1.0))
    rough = b.math("SUBTRACT", b.math("ADD", 0.78, b.math("MULTIPLY", fine, 0.18)), b.math("MULTIPLY", wet, 0.6))
    height = b.math("ADD", b.math("MULTIPLY", strata, 0.45), b.math("MULTIPLY", fine, 0.35))
    height = b.math("SUBTRACT", height, b.math("MULTIPLY", crack_m, 0.35))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.8
    bump.inputs["Distance"].default_value = 0.15
    b.link(b.math("ADD", height, b.math("MULTIPLY", moss_m, 0.2)), bump.inputs["Height"])
    p = principled(b, base=col, rough=rough, normal=bump.outputs["Normal"], spec=0.35)
    disp = b.n("ShaderNodeDisplacement")
    disp.inputs["Scale"].default_value = 0.6
    disp.inputs["Midlevel"].default_value = 0.5
    b.link(height, disp.inputs["Height"])
    b.link(p.outputs[0], out.inputs["Surface"])
    b.link(disp.outputs[0], out.inputs["Displacement"])
    set_displacement(mat)
    return mat


def mat_boulder():
    mat, b, out = new_material("MAT_Boulder_Granite")
    oi = b.n("ShaderNodeObjectInfo")
    tc = b.n("ShaderNodeTexCoord").outputs["Object"]
    rnd = oi.outputs["Random"]
    pos = b.vmath("ADD", tc, b.vmath("MULTIPLY", b.xyz(rnd, rnd, rnd), (37.0, 53.0, 71.0)))   # unique per boulder
    geo = b.n("ShaderNodeNewGeometry")
    nz = b.sep(geo.outputs["Normal"])[2]
    wz = b.sep(geo.outputs["Position"])[2]
    mott = b.noise(pos, 1.2, 6, 0.6).outputs["Fac"]
    speck = b.voronoi(pos, 35.0).outputs["Distance"]
    col = b.mix(mott, lin("#8c8a84"), lin("#b3b0a8"))
    col = b.mix(b.math("MULTIPLY", b.ramp_f(speck, 0.12, 0.0), 0.7), col, lin("#3c3a37"))
    col = b.mix(b.math("MULTIPLY", b.ramp_f(speck, 0.9, 1.0), 0.4), col, lin("#e6e2d8"))
    dirt = b.ramp_f(b.noise(pos, 2.5, 4).outputs["Fac"], 0.55, 0.7)
    col = b.mix(b.math("MULTIPLY", dirt, 0.5), col, lin("#6b5d47"))
    lich = b.ramp_f(b.noise(pos, 3.0, 5, 0.7).outputs["Fac"], 0.66, 0.72)
    col = b.mix(b.math("MULTIPLY", lich, 0.5), col, lin("#c9b37a"))
    moss_m = b.math("MULTIPLY", b.ramp_f(nz, 0.5, 0.85), b.ramp_f(mott, 0.4, 0.6))
    moss = b.mix(b.noise(pos, 25.0, 3).outputs["Fac"], lin("#2f4614"), lin("#58722a"))
    col = b.mix(moss_m, col, moss)
    wet = b.ramp_f(wz, 0.45, 0.05)
    col = b.mix(b.math("MULTIPLY", wet, 0.5), col, (0.0, 0.0, 0.0, 1.0))
    rough = b.math("SUBTRACT", b.math("ADD", 0.7, b.math("MULTIPLY", moss_m, 0.25)), b.math("MULTIPLY", wet, 0.55))
    fine = b.noise(pos, 14.0, 6).outputs["Fac"]
    height = b.math("ADD", b.math("MULTIPLY", mott, 0.6), b.math("MULTIPLY", fine, 0.25))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.45
    bump.inputs["Distance"].default_value = 0.02
    b.link(b.math("ADD", height, b.math("MULTIPLY", moss_m, b.math("MULTIPLY", fine, 0.8))), bump.inputs["Height"])
    p = principled(b, base=col, rough=rough, normal=bump.outputs["Normal"], spec=0.4)
    disp = b.n("ShaderNodeDisplacement")
    disp.inputs["Scale"].default_value = 0.06
    disp.inputs["Midlevel"].default_value = 0.5
    b.link(height, disp.inputs["Height"])
    b.link(p.outputs[0], out.inputs["Surface"])
    b.link(disp.outputs[0], out.inputs["Displacement"])
    set_displacement(mat)
    return mat


def mat_mountain():
    mat, b, out = new_material("MAT_Mountain")
    pos = b.n("ShaderNodeTexCoord").outputs["Object"]
    geo = b.n("ShaderNodeNewGeometry")
    nz = b.sep(geo.outputs["Normal"])[2]
    wz = b.sep(geo.outputs["Position"])[2]
    n1 = b.noise(pos, 0.02, 8, 0.6).outputs["Fac"]
    rock = b.mix(n1, lin("#5e626b"), lin("#8a8a8c"))
    forest = b.mix(n1, lin("#1c2a17"), lin("#2c3a20"))
    col = b.mix(b.ramp_f(b.math("ADD", wz, b.math("MULTIPLY", n1, 40.0)), 70.0, 110.0), forest, rock)
    snow_m = b.math("MULTIPLY", b.ramp_f(b.math("ADD", wz, b.math("MULTIPLY", n1, 60.0)), 140.0, 170.0), b.ramp_f(nz, 0.45, 0.7))
    col = b.mix(snow_m, col, lin("#eef2f6"))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 1.0
    bump.inputs["Distance"].default_value = 4.0
    b.link(b.noise(pos, 0.08, 10, 0.65).outputs["Fac"], bump.inputs["Height"])
    p = principled(b, base=col, rough=b.math("SUBTRACT", 0.9, b.math("MULTIPLY", snow_m, 0.4)), normal=bump.outputs["Normal"])
    b.link(p.outputs[0], out.inputs["Surface"])
    return mat


def mat_water():
    mat, b, out = new_material("MAT_Water_Glacial")
    pos = b.n("ShaderNodeTexCoord").outputs["Object"]
    mp = b.n("ShaderNodeMapping")
    b.link(pos, mp.inputs["Vector"])
    loc_fc = mp.inputs["Location"].driver_add("default_value", 1)
    loc_fc.driver.expression = "frame * 0.045"                 # flow toward the camera (-Y in object space)
    w = b.n("ShaderNodeValue")
    wfc = w.outputs[0].driver_add("default_value")
    wfc.driver.expression = "frame * 0.012"
    r1 = b.noise(mp.outputs[0], 0.9, 4, 0.55, w=w.outputs[0]).outputs["Fac"]
    r2 = b.noise(mp.outputs[0], 5.0, 3, 0.5, w=b.math("MULTIPLY", w.outputs[0], 2.3)).outputs["Fac"]
    h = b.math("ADD", b.math("MULTIPLY", r1, 0.7), b.math("MULTIPLY", r2, 0.3))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.35
    bump.inputs["Distance"].default_value = 0.05
    b.link(h, bump.inputs["Height"])
    water = principled(b, base=(1.0, 1.0, 1.0, 1.0), rough=0.04, ior=1.333, trans=1.0, spec=0.5, normal=bump.outputs["Normal"])
    foam_attr = b.n("ShaderNodeAttribute", attribute_name="foam", attribute_type="GEOMETRY").outputs["Fac"]
    fn = b.noise(mp.outputs[0], 3.0, 5, 0.6, w=w.outputs[0]).outputs["Fac"]
    foam = b.math("MULTIPLY", foam_attr, b.ramp_f(fn, 0.45, 0.62), clamp=True)
    foam_sh = principled(b, base=(0.92, 0.95, 0.96, 1.0), rough=0.55, sss=0.3, normal=bump.outputs["Normal"])
    mixs = b.n("ShaderNodeMixShader")
    b.link(foam, mixs.inputs["Fac"])
    b.link(water.outputs[0], mixs.inputs[1])
    b.link(foam_sh.outputs[0], mixs.inputs[2])
    b.link(mixs.outputs[0], out.inputs["Surface"])
    absorb = b.n("ShaderNodeVolumeAbsorption")
    absorb.inputs["Color"].default_value = (0.30, 0.78, 0.74, 1.0)
    absorb.inputs["Density"].default_value = 0.9
    scat = b.n("ShaderNodeVolumeScatter")
    scat.inputs["Color"].default_value = (0.55, 0.92, 0.90, 1.0)
    scat.inputs["Density"].default_value = 0.06
    scat.inputs["Anisotropy"].default_value = 0.3
    add = b.n("ShaderNodeAddShader")
    b.link(absorb.outputs[0], add.inputs[0])
    b.link(scat.outputs[0], add.inputs[1])
    b.link(add.outputs[0], out.inputs["Volume"])
    return mat


def mat_waterfall(speed):
    mat, b, out = new_material(f"MAT_Waterfall_{speed}")
    uv = b.n("ShaderNodeTexCoord").outputs["UV"]
    mp = b.n("ShaderNodeMapping")
    b.link(uv, mp.inputs["Vector"])
    mp.inputs["Scale"].default_value = (18.0, 2.2, 1.0)
    fc = mp.inputs["Location"].driver_add("default_value", 1)
    fc.driver.expression = f"-frame * {0.11 * speed:.4f}"      # streaks fall down (v grows downward)
    w = b.n("ShaderNodeValue")
    wf_ = w.outputs[0].driver_add("default_value")
    wf_.driver.expression = "frame * 0.02"
    n1 = b.noise(mp.outputs[0], 1.0, 6, 0.6, w=w.outputs[0]).outputs["Fac"]
    n2 = b.noise(mp.outputs[0], 4.0, 4, 0.6, w=w.outputs[0]).outputs["Fac"]
    streak = b.math("ADD", b.math("MULTIPLY", n1, 0.65), b.math("MULTIPLY", n2, 0.35))
    u, v, _ = b.sep(uv)
    edge = b.math("MULTIPLY", b.ramp_f(u, 0.0, 0.12), b.ramp_f(u, 1.0, 0.88))
    alpha = b.math("MULTIPLY", edge, b.ramp_f(b.math("ADD", streak, b.math("MULTIPLY", v, 0.25)), 0.35, 0.62))
    col = b.mix(streak, (0.70, 0.80, 0.82, 1.0), (0.97, 0.98, 0.98, 1.0))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.5
    b.link(streak, bump.inputs["Height"])
    p = principled(b, base=col, rough=0.3, ior=1.333, trans=0.35, sss=0.35, sss_scale=0.05, normal=bump.outputs["Normal"])
    tr = b.n("ShaderNodeBsdfTransparent")
    mixs = b.n("ShaderNodeMixShader")
    b.link(alpha, mixs.inputs["Fac"])
    b.link(tr.outputs[0], mixs.inputs[1])
    b.link(p.outputs[0], mixs.inputs[2])
    b.link(mixs.outputs[0], out.inputs["Surface"])
    try:
        mat.blend_method = "HASHED"
    except Exception:
        pass
    return mat


def mat_volume(name, density, color, aniso, noise_mist=False):
    mat, b, out = new_material(name)
    vol = b.n("ShaderNodeVolumePrincipled")
    vol.inputs["Color"].default_value = color
    vol.inputs["Anisotropy"].default_value = aniso
    if noise_mist:
        pos = b.n("ShaderNodeTexCoord").outputs["Generated"]
        w = b.n("ShaderNodeValue")
        fc = w.outputs[0].driver_add("default_value")
        fc.driver.expression = "frame * 0.015"
        mp = b.n("ShaderNodeMapping")
        b.link(pos, mp.inputs["Vector"])
        lfc = mp.inputs["Location"].driver_add("default_value", 2)
        lfc.driver.expression = "-frame * 0.002"            # mist drifts upward
        nz = b.noise(mp.outputs[0], 3.0, 4, 0.6, w=w.outputs[0]).outputs["Fac"]
        gz = b.sep(pos)[2]
        dens = b.math("MULTIPLY", b.ramp_f(nz, 0.42, 0.7), b.ramp_f(gz, 0.9, 0.05))
        b.link(b.math("MULTIPLY", dens, density), vol.inputs["Density"])
    else:
        vol.inputs["Density"].default_value = density
    b.link(vol.outputs[0], out.inputs["Volume"])
    return mat


def mat_leaf(name, deep, fresh, glow=0.38, dry_amt=0.45, hole_amt=0.12):
    """Thin-leaf PBR: per-leaf colour, veins, spots, dry edges, insect holes, translucency, waxy coat."""
    mat, b, out = new_material(name)
    uv = b.n("ShaderNodeTexCoord").outputs["UV"]
    u, v, _ = b.sep(uv)
    oi = b.n("ShaderNodeObjectInfo")
    rnd = oi.outputs["Random"]
    rnd2 = b.math("FRACT", b.math("MULTIPLY", rnd, 17.31))
    rnd3 = b.math("FRACT", b.math("MULTIPLY", rnd, 53.7))
    cu = b.math("ABSOLUTE", b.math("SUBTRACT", u, 0.5))                 # 0 at midrib, 0.5 at edge
    edge = b.math("MULTIPLY", cu, 2.0)
    t = b.math("ADD", b.math("ADD", b.math("MULTIPLY", v, 0.5), b.math("MULTIPLY", edge, 0.3)),
               b.math("MULTIPLY", b.math("SUBTRACT", rnd, 0.5), 0.6))
    col = b.ramp(t, [(0.0, deep), (0.55, b_mid(deep, fresh)), (1.0, fresh)])
    hs = b.n("ShaderNodeHueSaturation")
    b.link(col, hs.inputs["Color"])
    b.link(b.math("ADD", 0.5, b.math("MULTIPLY", b.math("SUBTRACT", rnd2, 0.5), 0.05)), hs.inputs["Hue"])
    b.link(b.math("ADD", 0.82, b.math("MULTIPLY", rnd3, 0.3)), hs.inputs["Value"])
    col = hs.outputs["Color"]
    # veins: midrib + lateral veins
    midrib = b.ramp_f(cu, 0.035, 0.0)
    lat = b.math("SINE", b.math("MULTIPLY", b.math("SUBTRACT", b.math("MULTIPLY", v, 13.0), b.math("MULTIPLY", cu, 5.0)), 6.2832))
    lat = b.math("MULTIPLY", b.ramp_f(lat, 0.85, 1.0), b.ramp_f(cu, 0.45, 0.2))
    veins = b.math("MAXIMUM", midrib, b.math("MULTIPLY", lat, 0.6))
    col = b.mix(b.math("MULTIPLY", veins, 0.35), col, b.mix(0.5, col, (0.85, 0.9, 0.55, 1.0)))
    # spots on some leaves
    spots = b.ramp_f(b.voronoi(uv, 22.0).outputs["Distance"], 0.09, 0.03)
    spot_on = b.ramp_f(rnd2, 0.72, 0.8)
    col = b.mix(b.math("MULTIPLY", spots, spot_on), col, lin("#4a3216"))
    # dry edges
    dry = b.math("MULTIPLY", b.ramp_f(edge, 0.72, 0.98), b.ramp_f(b.noise(uv, 7.0, 4).outputs["Fac"], 0.4, 0.6))
    dry = b.math("MULTIPLY", dry, b.math("MULTIPLY", b.ramp_f(rnd3, 0.55, 0.75), dry_amt * 2.0))
    col = b.mix(b.math("MINIMUM", dry, 1.0), col, lin("#8c6a2a"))
    # insect holes (alpha) on a few leaves
    holes = b.ramp_f(b.voronoi(uv, 7.0).outputs["Distance"], 0.13, 0.10)
    hole_on = b.ramp_f(rnd3, 1.0 - hole_amt, 1.0 - hole_amt + 0.01)
    alpha = b.math("SUBTRACT", 1.0, b.math("MULTIPLY", holes, hole_on), clamp=True)
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.35
    bump.inputs["Distance"].default_value = 0.002
    b.link(b.math("SUBTRACT", 1.0, veins), bump.inputs["Height"])
    p = principled(b, base=col, rough=0.42, spec=0.5, sss=0.12, sss_scale=0.002, coat=0.12, coat_rough=0.3,
                   normal=bump.outputs["Normal"])
    tl = b.n("ShaderNodeBsdfTranslucent")
    b.link(b.mix(0.25, col, (0.85, 0.85, 0.3, 1.0)), tl.inputs["Color"])
    b.link(bump.outputs["Normal"], tl.inputs["Normal"])
    m1 = b.n("ShaderNodeMixShader")
    m1.inputs["Fac"].default_value = glow
    b.link(p.outputs[0], m1.inputs[1])
    b.link(tl.outputs[0], m1.inputs[2])
    tr = b.n("ShaderNodeBsdfTransparent")
    m2 = b.n("ShaderNodeMixShader")
    b.link(alpha, m2.inputs["Fac"])
    b.link(tr.outputs[0], m2.inputs[1])
    b.link(m1.outputs[0], m2.inputs[2])
    b.link(m2.outputs[0], out.inputs["Surface"])
    return mat


def b_mid(a, c):
    return tuple((a[i] + c[i]) * 0.5 for i in range(3)) + (1.0,)


def mat_grass():
    mat, b, out = new_material("MAT_Grass")
    uv = b.n("ShaderNodeTexCoord").outputs["UV"]
    _, v, _ = b.sep(uv)
    oi = b.n("ShaderNodeObjectInfo")
    rnd = oi.outputs["Random"]
    col = b.ramp(v, [(0.0, lin("#1f3a0c")), (0.45, lin("#4f8a1c")), (0.85, lin("#86b83a")), (1.0, lin("#b9c96a"))])
    hs = b.n("ShaderNodeHueSaturation")
    b.link(col, hs.inputs["Color"])
    b.link(b.math("ADD", 0.48, b.math("MULTIPLY", rnd, 0.05)), hs.inputs["Hue"])
    b.link(b.math("ADD", 0.8, b.math("MULTIPLY", b.math("FRACT", b.math("MULTIPLY", rnd, 11.0)), 0.35)), hs.inputs["Value"])
    col = b.mix(b.ramp_f(b.math("FRACT", b.math("MULTIPLY", rnd, 37.0)), 0.93, 0.95), hs.outputs["Color"], lin("#b9a463"))
    p = principled(b, base=col, rough=0.45, spec=0.45, sss=0.08, sss_scale=0.002)
    tl = b.n("ShaderNodeBsdfTranslucent")
    b.link(col, tl.inputs["Color"])
    m = b.n("ShaderNodeMixShader")
    m.inputs["Fac"].default_value = 0.3
    b.link(p.outputs[0], m.inputs[1])
    b.link(tl.outputs[0], m.inputs[2])
    b.link(m.outputs[0], out.inputs["Surface"])
    return mat


def mat_needles():
    mat, b, out = new_material("MAT_Conifer_Needles")
    oi = b.n("ShaderNodeObjectInfo")
    rnd = oi.outputs["Random"]
    col = b.mix(rnd, lin("#16301a"), lin("#2d4a24"))
    p = principled(b, base=col, rough=0.5, spec=0.45, coat=0.08)
    tl = b.n("ShaderNodeBsdfTranslucent")
    b.link(col, tl.inputs["Color"])
    m = b.n("ShaderNodeMixShader")
    m.inputs["Fac"].default_value = 0.18
    b.link(p.outputs[0], m.inputs[1])
    b.link(tl.outputs[0], m.inputs[2])
    b.link(m.outputs[0], out.inputs["Surface"])
    return mat


def mat_bark(name="MAT_Bark", tint="#4b4036"):
    mat, b, out = new_material(name)
    pos = b.n("ShaderNodeTexCoord").outputs["Object"]
    mp = b.n("ShaderNodeMapping")
    b.link(pos, mp.inputs["Vector"])
    mp.inputs["Scale"].default_value = (6.0, 6.0, 0.9)
    fiss = b.voronoi(mp.outputs[0], 3.0, "DISTANCE_TO_EDGE").outputs["Distance"]
    n = b.noise(pos, 8.0, 6).outputs["Fac"]
    col = b.mix(n, lin(tint), lin("#7d7264"))
    col = b.mix(b.ramp_f(fiss, 0.08, 0.0), col, lin("#1d1915"))
    moss = b.ramp_f(b.noise(pos, 1.5, 4).outputs["Fac"], 0.55, 0.65)
    col = b.mix(b.math("MULTIPLY", moss, 0.6), col, lin("#4b6122"))
    bump = b.n("ShaderNodeBump")
    bump.inputs["Strength"].default_value = 0.9
    bump.inputs["Distance"].default_value = 0.02
    b.link(b.math("ADD", fiss, b.math("MULTIPLY", n, 0.3)), bump.inputs["Height"])
    p = principled(b, base=col, rough=0.85, normal=bump.outputs["Normal"])
    b.link(p.outputs[0], out.inputs["Surface"])
    return mat


def mat_simple(name, color, rough=0.8):
    mat, b, out = new_material(name)
    p = principled(b, base=color, rough=rough)
    b.link(p.outputs[0], out.inputs["Surface"])
    return mat


# ------------------------------------------------------------------ ASSET MESHES (real geometry)
def leaf_geometry(seed, width=0.16, fold=0.10, curl=0.18, crumple=0.025, ovate=0.0, rows=14, cols=6):
    """Unit-length leaf: stem at origin, length along +Y, width along X, folded at the midrib,
    curled along its length and slightly crumpled."""
    rnd = random.Random(seed)
    verts, uvs = [], []
    for j in range(rows + 1):
        v = j / rows
        prof = math.sin(math.pi * min(1.0, v * (1.0 + ovate))) ** 0.85 * (1.0 - 0.35 * v * (1 - ovate))
        w = width * max(prof, 0.0) + 0.004
        for i in range(cols + 1):
            u = i / cols
            x = (u - 0.5) * 2 * w
            z = fold * abs(x) / max(width, 1e-3) * 0.5        # V-fold along the midrib
            z += curl * v * v                                 # curl along the length
            z += crumple * wf.vnoise(x * 40 + seed, v * 25, 0.5, seed) * (0.3 + v)
            y = v + 0.02 * wf.vnoise(x * 30, v * 10, 1.5, seed + 1)
            verts.append((x, y, z))
            uvs.append((u, v))
    faces = [(j * (cols + 1) + i, j * (cols + 1) + i + 1, (j + 1) * (cols + 1) + i + 1, (j + 1) * (cols + 1) + i)
             for j in range(rows) for i in range(cols)]
    rnd.random()
    return verts, faces, uvs


def grass_tuft_geometry(seed, blades=24, hmin=0.15, hmax=0.45, wbase=0.005, spread=0.06):
    rnd = random.Random(seed)
    verts, faces, uvs = [], [], []
    for _ in range(blades):
        a = rnd.uniform(0, math.tau)
        ox, oy = math.cos(a) * rnd.uniform(0, spread), math.sin(a) * rnd.uniform(0, spread)
        h = rnd.uniform(hmin, hmax)
        lean = rnd.uniform(0.15, 0.55) * h
        dx, dy = math.cos(a + rnd.uniform(-0.6, 0.6)), math.sin(a + rnd.uniform(-0.6, 0.6))
        px, py = -dy, dx
        twist = rnd.uniform(-0.6, 0.6)
        base = len(verts)
        segs = 5
        for s in range(segs + 1):
            t = s / segs
            w = wbase * (1 - t) ** 0.9 + 0.0004
            cx, cy, cz = ox + dx * lean * t * t, oy + dy * lean * t * t, h * (t - 0.18 * t * t * lean / h)
            ca, sa = math.cos(twist * t), math.sin(twist * t)
            qx, qy = px * ca - py * sa, px * sa + py * ca
            verts += [(cx - qx * w, cy - qy * w, cz), (cx + qx * w, cy + qy * w, cz)]
            uvs += [(0.0, t), (1.0, t)]
        for s in range(segs):
            i0 = base + 2 * s
            faces.append((i0, i0 + 1, i0 + 3, i0 + 2))
    return verts, faces, uvs


def needle_cluster_geometry(seed, needles=34, nlen=0.12, nw=0.006):
    """A fir/spruce twig: axis along +Y (length 1), needles spiralling around it."""
    rnd = random.Random(seed)
    verts, faces = [], []
    for k in range(needles):
        t = 0.05 + 0.9 * k / needles
        th = k * 2.399 + rnd.uniform(-0.2, 0.2)
        d = Vector((math.cos(th) * 0.85, 0.45 + rnd.uniform(-0.1, 0.2), math.sin(th) * 0.55 + 0.25)).normalized()
        side = d.cross(Vector((0, 1, 0))).normalized() * nw
        p0 = Vector((0, t, 0))
        p1 = p0 + d * nlen * rnd.uniform(0.75, 1.15)
        b = len(verts)
        verts += [tuple(p0 - side), tuple(p0 + side), tuple(p1 + side * 0.2), tuple(p1 - side * 0.2)]
        faces.append((b, b + 1, b + 2, b + 3))
    b = len(verts)                                                # twig
    verts += [(-0.006, 0, 0), (0.006, 0, 0), (0.002, 1.0, 0), (-0.002, 1.0, 0)]
    faces.append((b, b + 1, b + 2, b + 3))
    return verts, faces


def euler_from_dir(d, roll):
    q = Vector(d).normalized().to_track_quat("Y", "Z") @ Quaternion((0, 1, 0), roll)
    e = q.to_euler("XYZ")
    return (e.x, e.y, e.z)


# ------------------------------------------------------------------ GEOMETRY NODES
def gn_interface(ng, inputs, with_geo_out=True):
    ng.interface.new_socket(name="Geometry", in_out="INPUT", socket_type="NodeSocketGeometry")
    for name, stype in inputs:
        ng.interface.new_socket(name=name, in_out="INPUT", socket_type=stype)
    if with_geo_out:
        ng.interface.new_socket(name="Geometry", in_out="OUTPUT", socket_type="NodeSocketGeometry")


def set_mod_inputs(mod, values):
    for item in mod.node_group.interface.items_tree:
        if getattr(item, "item_type", "SOCKET") == "SOCKET" and item.in_out == "INPUT" and item.name in values:
            mod[item.identifier] = values[item.name]


def to_rotation(b, vec):
    try:
        node = b.n("FunctionNodeEulerToRotation")
        b.link(vec, node.inputs[0])
        return node.outputs[0]
    except RuntimeError:
        return vec


def random_value(b, dtype, lo, hi, seed):
    node = b.n("FunctionNodeRandomValue", data_type=dtype)
    idx = {"FLOAT_VECTOR": (0, 1, 0), "FLOAT": (2, 3, 1), "INT": (4, 5, 2)}[dtype]
    node.inputs[idx[0]].default_value = lo
    node.inputs[idx[1]].default_value = hi
    node.inputs["Seed"].default_value = seed
    return node.outputs[idx[2]]


def named(b, name, dtype):
    node = b.n("GeometryNodeInputNamedAttribute", data_type=dtype)
    node.inputs["Name"].default_value = name
    return node.outputs[0]


def wave(b, t, omega, phase, amp):
    return b.math("MULTIPLY", b.math("SINE", b.math("ADD", b.math("MULTIPLY", t, omega), phase)), amp)


def gn_scatter_group():
    """Points with attributes rot/scl/phase/amp -> swaying, fluttering instances from a collection."""
    ng = bpy.data.node_groups.new("GN_WindInstances", "GeometryNodeTree")
    gn_interface(ng, [("Instances", "NodeSocketCollection"), ("Wind", "NodeSocketFloat")])
    b = NB(ng)
    gi, go = b.n("NodeGroupInput"), b.n("NodeGroupOutput")
    t = b.n("GeometryNodeInputSceneTime").outputs["Seconds"]
    ph = named(b, "phase", "FLOAT")
    amp = b.math("MULTIPLY", named(b, "amp", "FLOAT"), gi.outputs["Wind"])
    w1, w2, w3 = 2 * math.pi * 3 / 20, 2 * math.pi * 8 / 20, 2 * math.pi * 2 / 20     # loops every 20 s
    ox = b.math("MULTIPLY", b.math("ADD", wave(b, t, w1, ph, 0.6), wave(b, t, w2, b.math("MULTIPLY", ph, 1.7), 0.22)), amp)
    oy = b.math("MULTIPLY", wave(b, t, w3, b.math("MULTIPLY", ph, 0.7), 0.45), amp)
    sp = b.n("GeometryNodeSetPosition")
    b.link(gi.outputs["Geometry"], sp.inputs["Geometry"])
    b.link(b.xyz(ox, oy, 0.0), sp.inputs["Offset"])
    ci = b.n("GeometryNodeCollectionInfo")
    b.link(gi.outputs["Instances"], ci.inputs["Collection"])
    ci.inputs["Separate Children"].default_value = True
    ci.inputs["Reset Children"].default_value = True
    iop = b.n("GeometryNodeInstanceOnPoints")
    b.link(sp.outputs[0], iop.inputs["Points"])
    b.link(ci.outputs[0], iop.inputs["Instance"])
    iop.inputs["Pick Instance"].default_value = True
    b.link(random_value(b, "INT", 0, 99, 11), iop.inputs["Instance Index"])
    b.link(to_rotation(b, named(b, "rot", "FLOAT_VECTOR")), iop.inputs["Rotation"])
    scl = named(b, "scl", "FLOAT")
    b.link(b.xyz(scl, scl, scl), iop.inputs["Scale"])
    flutter = b.math("MULTIPLY", wave(b, t, 2 * math.pi * 22 / 20, b.math("MULTIPLY", ph, 5.0), 0.14), gi.outputs["Wind"])
    rot = b.n("GeometryNodeRotateInstances")
    b.link(iop.outputs[0], rot.inputs["Instances"])
    b.link(to_rotation(b, b.xyz(flutter, 0.0, 0.0)), rot.inputs["Rotation"])
    rot.inputs["Local Space"].default_value = True
    b.link(rot.outputs[0], go.inputs[0])
    return ng


def gn_grass_group():
    ng = bpy.data.node_groups.new("GN_GrassScatter", "GeometryNodeTree")
    gn_interface(ng, [("Ground", "NodeSocketObject"), ("Tufts", "NodeSocketCollection"),
                      ("Wild", "NodeSocketCollection"), ("Density", "NodeSocketFloat")])
    b = NB(ng)
    gi, go = b.n("NodeGroupInput"), b.n("NodeGroupOutput")
    oi = b.n("GeometryNodeObjectInfo", transform_space="RELATIVE")
    b.link(gi.outputs["Ground"], oi.inputs["Object"])
    t = b.n("GeometryNodeInputSceneTime").outputs["Seconds"]
    join = b.n("GeometryNodeJoinGeometry")
    for attr, coll_name, dens_mul, seed, smin, smax in (("grass", "Tufts", 1.0, 3, 0.8, 1.2), ("wild", "Wild", 0.25, 7, 0.8, 1.25)):
        dist = b.n("GeometryNodeDistributePointsOnFaces", distribute_method="RANDOM")
        b.link(oi.outputs["Geometry"], dist.inputs["Mesh"])
        b.link(b.math("MULTIPLY", named(b, attr, "FLOAT"), b.math("MULTIPLY", gi.outputs["Density"], dens_mul)), dist.inputs["Density"])
        dist.inputs["Seed"].default_value = seed
        ci = b.n("GeometryNodeCollectionInfo")
        b.link(gi.outputs[coll_name], ci.inputs["Collection"])
        ci.inputs["Separate Children"].default_value = True
        ci.inputs["Reset Children"].default_value = True
        iop = b.n("GeometryNodeInstanceOnPoints")
        b.link(dist.outputs["Points"], iop.inputs["Points"])
        b.link(ci.outputs[0], iop.inputs["Instance"])
        iop.inputs["Pick Instance"].default_value = True
        b.link(random_value(b, "INT", 0, 99, seed + 1), iop.inputs["Instance Index"])
        b.link(to_rotation(b, b.xyz(0.0, 0.0, random_value(b, "FLOAT", 0.0, 6.283, seed + 2))), iop.inputs["Rotation"])
        s = random_value(b, "FLOAT", smin, smax, seed + 3)
        b.link(b.xyz(s, s, s), iop.inputs["Scale"])
        p = b.sep(b.n("GeometryNodeInputPosition").outputs[0])
        ph = b.math("ADD", b.math("MULTIPLY", p[0], 0.35), b.math("MULTIPLY", p[1], 0.21))
        sx = b.math("ADD", wave(b, t, 2 * math.pi * 5 / 20, ph, 0.09), wave(b, t, 2 * math.pi * 12 / 20, b.math("MULTIPLY", ph, 2.0), 0.035))
        sy = wave(b, t, 2 * math.pi * 4 / 20, b.math("MULTIPLY", ph, 0.8), 0.06)
        rot = b.n("GeometryNodeRotateInstances")
        b.link(iop.outputs[0], rot.inputs["Instances"])
        b.link(to_rotation(b, b.xyz(sx, sy, 0.0)), rot.inputs["Rotation"])
        b.link(rot.outputs[0], join.inputs[0])
    b.link(join.outputs[0], go.inputs[0])
    return ng


# ------------------------------------------------------------------ CURVES (trunks, branches, twigs)
def curve_obj(name, splines, coll, mat, resolution=3):
    """splines: list of (points[(x,y,z)], radii[])"""
    cu = bpy.data.curves.new(name, "CURVE")
    cu.dimensions = "3D"
    cu.bevel_depth = 1.0
    cu.bevel_resolution = resolution
    cu.use_fill_caps = True
    for pts, radii in splines:
        sp = cu.splines.new("POLY")
        sp.points.add(len(pts) - 1)
        for p, r, cp in zip(pts, radii, sp.points):
            cp.co = (p[0], p[1], p[2], 1.0)
            cp.radius = r
    obj = bpy.data.objects.new(name, cu)
    coll.objects.link(obj)
    cu.materials.append(mat)
    return obj


def catmull3(pts, samples):
    out = []
    n = len(pts)
    for i in range(n - 1):
        p0, p1, p2, p3 = pts[max(i - 1, 0)], pts[i], pts[i + 1], pts[min(i + 2, n - 1)]
        for k in range(samples):
            t = k / samples
            t2, t3 = t * t, t * t * t
            out.append(Vector(tuple(0.5 * ((2 * p1[c]) + (-p0[c] + p2[c]) * t + (2 * p0[c] - 5 * p1[c] + 4 * p2[c] - p3[c]) * t2
                                           + (-p0[c] + 3 * p1[c] - 3 * p2[c] + p3[c]) * t3) for c in range(3))))
    out.append(Vector(pts[-1]))
    return out


# ------------------------------------------------------------------ BUILD
def build():
    wf.USE_CLAY_OVERRIDE = False
    wf.SAVE_BLEND = False
    wf.RENDER_STILL_IN_BACKGROUND = False
    wf.main()
    scene = bpy.context.scene
    vl = bpy.context.view_layer
    vl.material_override = None
    C = {c.name: c for c in bpy.data.collections}
    rnd = random.Random(2026)

    assets = collection("_Assets")

    # --- remove blockout proxies, remember their placement
    pines, bushes = [], []
    for o in list(bpy.data.objects):
        if o.name.startswith("Pine_"):
            pines.append((Vector(o.location), o.scale.z))
            bpy.data.objects.remove(o, do_unlink=True)
        elif o.name.startswith("Bush_"):
            bushes.append((Vector(o.location), o.scale.y))
            bpy.data.objects.remove(o, do_unlink=True)
        elif o.name == "River_Surface":
            bpy.data.objects.remove(o, do_unlink=True)
    for me in list(bpy.data.meshes):
        if me.users == 0:
            bpy.data.meshes.remove(me)

    # --- terrain: attributes (river edge distance, grass density) + ground material
    terrain = bpy.data.objects["Terrain_Valley"]
    me = terrain.data
    cam_path = [(0, -2), (mx(3.5), 24), (mx(8), 40)]
    e_vals, g_vals, w_vals = [], [], []
    for vert in me.vertices:
        x, y, z = vert.co
        d, rw = wf.river_info(x, y)
        e = d - rw
        e_vals.append(e)
        visible = abs(x) < 28 + max(y, 0) * 0.65 and -6 < y < 110
        dcam = min(math.hypot(x - cx, y - cy) for cx, cy in cam_path)
        fall = max(0.12, 1.0 - dcam / 90.0)
        slope_ok = vert.normal.z > 0.72
        g = (1.0 if (e > 1.6 and slope_ok and visible and z > 0.15) else 0.0) * fall
        g_vals.append(g)
        w_vals.append(1.0 if (1.0 < e < 4.5 and visible and y < 95) else 0.0)
    for name, vals in (("e", e_vals), ("grass", g_vals), ("wild", w_vals)):
        a = me.attributes.new(name, "FLOAT", "POINT")
        a.data.foreach_set("value", vals)
    assign(terrain, mat_terrain())
    subsurf(terrain, 2)

    # --- cliffs, rocks, mountains
    fall_xy = (mx(44), 100)
    cliff_mat = mat_cliff(fall_xy)
    for name in ("Cliff_Left_Centre", "Cliff_Right"):
        o = bpy.data.objects[name]
        assign(o, cliff_mat)
        subsurf(o, 2)
    rock_mat = mat_boulder()
    for o in bpy.data.objects:
        if o.name.startswith(("Boulder", "Pebble")):
            if not o.data.materials or o.data.materials[0] != rock_mat:
                assign(o, rock_mat)
            subsurf(o, 2 if o.name.startswith("Boulder_Hero") else 1, simple=False)
    assign(bpy.data.objects["Terrain_Mountains"], mat_mountain())

    # --- water: grid with foam attribute, solidified so the glacial volume works
    rocks_in_water = []
    for o in bpy.data.objects:
        if o.name.startswith("Boulder"):
            dims = o.dimensions
            if o.location.z - dims.z * 0.3 < 0.2 < o.location.z + dims.z * 0.6:
                rocks_in_water.append((o.location.x, o.location.y, max(dims.x, dims.y) * 0.5))
    wfm = bpy.data.objects["Waterfall_Main"]
    zmin = min(vv.co.z for vv in wfm.data.vertices)
    low = [vv.co for vv in wfm.data.vertices if vv.co.z < zmin + 0.3]
    fb = wfm.matrix_world @ (sum(low, Vector()) / len(low))     # base of the falls
    step, xs0, ys0, nx, ny = 0.5, -45.0, -30.0, 201, 271
    verts, foam, hgt = [], [], []
    for i in range(nx):
        for j in range(ny):
            x, y = xs0 + i * step, ys0 + j * step
            verts.append((x, y, 0.0))
            hgt.append(wf.terrain_h(x, y))
            f = 0.0
            for rx, ry, rr_ in rocks_in_water:
                f += max(0.0, 1.0 - max(0.0, math.hypot(x - rx, y - ry) - rr_ * 0.9) / 1.6)
            f += max(0.0, 1.0 - math.hypot(x - fb.x, y - fb.y) / 10.0) * 1.5
            d, rw = wf.river_info(x, y)
            f += max(0.0, 1.0 - abs(d - rw) / 1.0) * 0.35
            foam.append(min(f, 1.0))
    faces = []
    for i in range(nx - 1):
        for j in range(ny - 1):
            q = (i * ny + j, (i + 1) * ny + j, (i + 1) * ny + j + 1, i * ny + j + 1)
            if min(hgt[k] for k in q) < 0.25:
                faces.append(q)
    water = mesh_obj("Water_River", verts, faces, C["Water"], smooth=True)
    water.data.attributes.new("foam", "FLOAT", "POINT").data.foreach_set("value", foam)
    sol = water.modifiers.new("Depth", "SOLIDIFY")
    sol.thickness = 2.2
    sol.offset = -1.0
    assign(water, mat_water())

    # --- waterfalls: UVs + animated material
    for name, cols, speed in (("Waterfall_Main", 24, 1.0), ("Waterfall_Strand", 6, 1.2)):
        o = bpy.data.objects[name]
        rows = len(o.data.vertices) // (cols + 1)
        uvs = [(i / cols, j / (rows - 1)) for i in range(cols + 1) for j in range(rows)]
        uvl = o.data.uv_layers.new(name="UVMap")
        for loop in o.data.loops:
            uvl.data[loop.index].uv = uvs[loop.vertex_index]
        assign(o, mat_waterfall(speed))
        subsurf(o, 1, simple=False)

    # --- atmosphere: valley haze + waterfall mist
    def box(name, cx, cy, cz, sx, sy, sz, mat, coll):
        vs = [(cx + dx * sx, cy + dy * sy, cz + dz * sz) for dx in (-0.5, 0.5) for dy in (-0.5, 0.5) for dz in (-0.5, 0.5)]
        fs = [(0, 1, 3, 2), (4, 6, 7, 5), (0, 4, 5, 1), (2, 3, 7, 6), (0, 2, 6, 4), (1, 5, 7, 3)]
        o = mesh_obj(name, vs, fs, coll, smooth=False)
        assign(o, mat)
        return o
    box("VOL_Haze", 0, 520, 200, 2000, 1200, 460, mat_volume("MAT_Haze", HAZE_DENSITY, (0.78, 0.86, 1.0, 1.0), 0.55), C["Lights"])
    box("VOL_FallMist", fb.x, fb.y - 3, 7, 34, 18, 20,
        mat_volume("MAT_Mist", 0.09, (1.0, 1.0, 1.0, 1.0), 0.3, noise_mist=True), C["Water"])

    # --- world: Nishita sky + clouds, sun lamp
    az, el = math.radians(mx(SUN_AZIMUTH)), math.radians(SUN_ELEVATION)
    sdir = Vector((math.sin(az) * math.cos(el), math.cos(az) * math.cos(el), math.sin(el)))
    world = bpy.data.worlds.new("WLD_Nishita")
    try:
        world.use_nodes = True
    except Exception:
        pass
    world.node_tree.nodes.clear()
    b = NB(world.node_tree)
    wout = b.n("ShaderNodeOutputWorld")
    sky = b.n("ShaderNodeTexSky")
    for st in ("NISHITA", "MULTIPLE_SCATTERING", "SINGLE_SCATTERING"):
        try:
            sky.sky_type = st
            break
        except TypeError:
            continue
    for k, v in (("sun_disc", False), ("sun_elevation", el), ("sun_rotation", -az), ("altitude", 800.0),
                 ("air_density", 1.0), ("dust_density", 1.6), ("ozone_density", 1.0)):
        try:
            setattr(sky, k, v)
        except Exception as ex:
            print("WARN sky", k, ex)
    dirv = b.n("ShaderNodeTexCoord").outputs["Generated"]
    dx, dy, dz = b.sep(dirv)
    den = b.math("ADD", b.math("MAXIMUM", dz, 0.0), 0.08)
    cp = b.xyz(b.math("DIVIDE", dx, den), b.math("DIVIDE", dy, den), 0.0)
    cl = b.noise(cp, 0.55, 6, 0.6).outputs["Fac"]
    cmask = b.math("MULTIPLY", b.ramp_f(cl, 0.56, 0.72), b.ramp_f(dz, 0.03, 0.25))
    bw = b.n("ShaderNodeRGBToBW")
    b.link(sky.outputs["Color"], bw.inputs["Color"])
    cloud_col = b.math("MULTIPLY", bw.outputs[0], 1.9)
    skycol = b.mix(b.math("MULTIPLY", cmask, 0.85), sky.outputs["Color"], b.xyz(cloud_col, cloud_col, cloud_col))
    bg = b.n("ShaderNodeBackground")
    b.link(skycol, bg.inputs["Color"])
    bg.inputs["Strength"].default_value = SKY_STRENGTH
    b.link(bg.outputs[0], wout.inputs["Surface"])
    scene.world = world
    sun = bpy.data.objects["LGT_Sun_Key"]
    sun.data.energy = SUN_STRENGTH
    sun.data.angle = math.radians(1.2)
    sun.data.color = (1.0, 0.93, 0.82)
    sun.rotation_euler = (-sdir).to_track_quat("-Z", "Y").to_euler()

    # --- asset collections
    def asset_coll(name):
        return collection(name, assets)
    leaf_tree = asset_coll("LeafTree")
    leaf_bush = asset_coll("LeafBush")
    tufts = asset_coll("GrassTufts")
    wild = asset_coll("GrassWild")
    needles = asset_coll("Needles")
    mat_lt = mat_leaf("MAT_Leaf_Weeping", lin("#4C8C2B"), lin("#B5D334"))
    mat_lb = mat_leaf("MAT_Leaf_Bush", lin("#2c5418"), lin("#6e9a2a"), glow=0.3, dry_amt=0.3, hole_amt=0.06)
    for k, (fold, curl, crum, wdt, ov) in enumerate([(0.10, 0.18, 0.02, 0.15, 0.0), (0.16, 0.30, 0.03, 0.13, 0.0),
                                                      (0.06, 0.10, 0.04, 0.17, 0.0), (0.12, -0.12, 0.03, 0.14, 0.0)]):
        v, f, uv = leaf_geometry(100 + k, wdt, fold, curl, crum, ov)
        o = mesh_obj(f"Leaf_Lance_{k}", v, f, leaf_tree, uvs=uv)
        s = o.modifiers.new("Thickness", "SOLIDIFY")
        s.thickness = 0.006
        s.offset = 0.0
        assign(o, mat_lt)
    for k, (fold, curl, crum, wdt, ov) in enumerate([(0.12, 0.15, 0.03, 0.30, 0.35), (0.18, 0.25, 0.04, 0.26, 0.25),
                                                      (0.08, -0.1, 0.03, 0.32, 0.4)]):
        v, f, uv = leaf_geometry(200 + k, wdt, fold, curl, crum, ov)
        o = mesh_obj(f"Leaf_Ovate_{k}", v, f, leaf_bush, uvs=uv)
        s = o.modifiers.new("Thickness", "SOLIDIFY")
        s.thickness = 0.008
        assign(o, mat_lb)
    gmat = mat_grass()
    for k in range(4):
        v, f, uv = grass_tuft_geometry(300 + k)
        assign(mesh_obj(f"GrassTuft_{k}", v, f, tufts, uvs=uv), gmat)
    for k in range(2):
        v, f, uv = grass_tuft_geometry(400 + k, blades=14, hmin=0.55, hmax=1.0, wbase=0.007, spread=0.1)
        assign(mesh_obj(f"GrassWild_{k}", v, f, wild, uvs=uv), gmat)
    nmat = mat_needles()
    for k in range(3):
        v, f = needle_cluster_geometry(500 + k)
        assign(mesh_obj(f"NeedleTwig_{k}", v, f, needles, smooth=False), nmat)
    bark = mat_bark()
    ng_wind = gn_scatter_group()

    def scatter(name, pts, rots, scls, phases, amps, inst_coll, coll, wind=1.0):
        o = point_cloud_obj(name, pts, {"rot": ("FLOAT_VECTOR", rots), "scl": ("FLOAT", scls),
                                        "phase": ("FLOAT", phases), "amp": ("FLOAT", amps)}, coll)
        mod = o.modifiers.new("Instances", "NODES")
        mod.node_group = ng_wind
        set_mod_inputs(mod, {"Instances": inst_coll, "Wind": wind})
        return o

    # --- foreground weeping tree (top-right)
    side = -1 if wf.MIRROR_X else 1
    bx = mx(19)
    g0 = wf.terrain_h(bx, 20)
    trunk_pts = [(bx, 20, g0 - 0.5), (bx - 0.5 * side, 20.4, g0 + 4), (bx - 2 * side, 21, g0 + 8), (bx - 4 * side, 21.5, g0 + 11)]
    trunk = catmull3(trunk_pts, 8)
    splines = [(trunk, [0.42 * (1 - 0.7 * i / len(trunk)) for i in range(len(trunk))])]
    twigs, L_pts, L_rot, L_scl, L_ph, L_amp = [], [], [], [], [], []
    trng = random.Random(77)
    rr = trng.uniform

    def add_leaf(p, d, scl, amp):
        L_pts.append(tuple(p))
        L_rot.append(euler_from_dir(d, rr(0, math.tau)))
        L_scl.append(scl)
        L_ph.append(rr(0, math.tau))
        L_amp.append(amp)

    for _ in range(11):
        s = trunk[int(rr(0.55, 0.999) * (len(trunk) - 1))]
        ang = rr(-0.9, 1.0)
        ln = rr(4.5, 8.0)
        d = Vector((-math.cos(ang) * side, math.sin(ang) * 0.9, 0.0)).normalized()
        p1 = s + d * ln * 0.4 + Vector((0, 0, rr(1.5, 3.0)))
        p2 = s + d * ln * 0.75 + Vector((0, 0, rr(1.0, 2.5)))
        p3 = s + d * ln + Vector((0, 0, rr(-0.5, 1.0)))
        br = catmull3([tuple(s), tuple(p1), tuple(p2), tuple(p3)], 8)
        r0 = rr(0.07, 0.14)
        splines.append((br, [r0 * (1 - 0.85 * i / len(br)) + 0.008 for i in range(len(br))]))
        for _ in range(LEAF_STRANDS_PER_BRANCH):
            t = rr(0.25, 1.0)
            a = br[int(t * (len(br) - 1))]
            L = rr(1.0, 2.2) + t * rr(1.5, 3.2)
            ph = rr(0, math.tau)
            drift = Vector((rr(-0.25, 0.25), rr(-0.25, 0.25), 0))
            n = int(L / 0.04)
            tw = []
            for k in range(n + 1):
                f = k / n
                p = a + Vector((drift.x * f * L + math.sin(f * 4 + ph) * 0.06, drift.y * f * L, -f * L))
                if k % 6 == 0:
                    tw.append(p)
                for _ in range(2):
                    out = Vector((rr(-1, 1), rr(-1, 1), 0)).normalized()
                    dleaf = (Vector((0, 0, -1)) + out * rr(0.4, 1.1)).normalized()
                    add_leaf(p + out * 0.01, dleaf, 0.05 * rr(0.7, 1.3), 0.03 + 0.22 * f * L / 4.5)
            if len(tw) >= 2:
                twigs.append((tw, [0.004] * len(tw)))
            for _ in range(10):
                q = a + Vector((rr(-0.6, 0.6), rr(-0.6, 0.6), rr(-0.3, 0.4)))
                add_leaf(q, Vector((rr(-1, 1), rr(-1, 1), rr(-1, 0.3))), 0.05 * rr(0.7, 1.3), 0.02)
    tree = collection("Tree_Foreground", C["Vegetation"])
    for o in (curve_obj("Tree_TrunkBranches", splines, tree, bark, 4), curve_obj("Tree_Twigs", twigs, tree, bark, 1)):
        o.data.bevel_resolution = 1 if o.name.endswith("Twigs") else 4
    scatter("Tree_Leaves", L_pts, L_rot, L_scl, L_ph, L_amp, leaf_tree, C["Leaves"])
    print("tree leaves:", len(L_pts))

    # --- bushes (leaf clumps + dark core)
    core_mat = mat_simple("MAT_BushCore", lin("#18260f"), 0.9)
    bp, br_, bs, bph, bam = [], [], [], [], []
    cv, cf = wf.icosphere(2)
    for (loc, sc) in bushes:
        r = 0.9 * sc
        core = mesh_obj("Bush_Core", [(x * r * 0.75, y * r * 0.75, max(z, -0.2) * r * 0.5 + r * 0.35) for x, y, z in cv], cf, C["Vegetation"])
        core.location = loc
        assign(core, core_mat)
        for _ in range(int(900 * r * r + 200)):
            u, vv = rr(0, math.tau), math.acos(rr(-0.2, 1.0))
            rad = r * rr(0.55, 1.0) ** 0.33
            off = Vector((math.cos(u) * math.sin(vv) * rad * 1.1, math.sin(u) * math.sin(vv) * rad * 1.1, math.cos(vv) * rad * 0.7 + r * 0.3))
            p = loc + off
            d = (off.normalized() + Vector((0, 0, 0.4)) + Vector((rr(-0.5, 0.5), rr(-0.5, 0.5), rr(-0.5, 0.5)))).normalized()
            bp.append(tuple(p)); br_.append(euler_from_dir(d, rr(0, math.tau)))
            bs.append(0.075 * rr(0.7, 1.3)); bph.append(rr(0, math.tau)); bam.append(0.025 * (off.z / (r + 0.01)))
    scatter("Bush_Leaves", bp, br_, bs, bph, bam, leaf_bush, C["Leaves"], wind=0.8)

    # --- conifers: 4 variants (trunk + whorled branches + needle twigs), placed as collection instances
    variants = []
    for k in range(4):
        vc = collection(f"PineVar_{k}", assets)
        prng = random.Random(900 + k)
        H = 16.0
        sp = [([(prng.uniform(-0.05, 0.05), prng.uniform(-0.05, 0.05), z * 0.5) for z in range(int(H / 0.5) + 1)],
               [0.34 * (1 - z * 0.5 / H) ** 0.8 + 0.02 for z in range(int(H / 0.5) + 1)])]
        npts, nrot, nscl = [], [], []
        z = 1.8
        while z < H - 0.4:
            f = z / H
            nb = prng.randint(5, 7)
            L = 4.4 * (1 - f) ** 0.9 + 0.35
            for i in range(nb):
                a = i / nb * math.tau + prng.uniform(-0.3, 0.3) + z * 1.3
                d = Vector((math.cos(a), math.sin(a), 0))
                pts = []
                for s in range(9):
                    t = s / 8
                    pts.append(Vector((0, 0, z)) + d * L * t + Vector((0, 0, -0.35 * L * t * t + 0.12 * L * t * f)))
                sp.append(([tuple(p) for p in pts], [0.05 * (1 - t / 8) + 0.006 for t in range(9)]))
                steps = max(2, int(L / 0.17))
                for s in range(steps):
                    t = (s + 0.5) / steps
                    p = pts[min(8, int(t * 8))]
                    tang = (pts[min(8, int(t * 8) + 1)] - pts[max(0, int(t * 8) - 1)]).normalized()
                    lat = tang.cross(Vector((0, 0, 1))).normalized()
                    for sgn in (-1, 0, 1):
                        dd = (tang + lat * sgn * 0.9 + Vector((0, 0, prng.uniform(-0.3, 0.1)))).normalized()
                        npts.append(tuple(p + lat * sgn * 0.05))
                        nrot.append(euler_from_dir(dd, prng.uniform(-0.6, 0.6)))
                        nscl.append(0.26 * prng.uniform(0.8, 1.3) * (1 - 0.3 * t))
            z += prng.uniform(0.38, 0.55)
        for _ in range(60):                                          # leader tip
            p = Vector((prng.uniform(-0.2, 0.2), prng.uniform(-0.2, 0.2), H - prng.uniform(0, 1.2)))
            npts.append(tuple(p))
            nrot.append(euler_from_dir(Vector((prng.uniform(-1, 1), prng.uniform(-1, 1), 1.2)), prng.uniform(0, 6)))
            nscl.append(0.2)
        curve_obj(f"PineVar_{k}_Wood", sp, vc, bark, 2)
        scatter(f"PineVar_{k}_Needles", npts, nrot, nscl, [0.0] * len(npts), [0.0] * len(npts), needles, vc, wind=0.0)
        variants.append(vc)
    for i, (loc, h) in enumerate(pines):
        e = bpy.data.objects.new(f"Pine_{i:03d}", None)
        e.instance_type = "COLLECTION"
        e.instance_collection = variants[i % 4]
        e.location = loc
        e.rotation_euler = (0, 0, rnd.uniform(0, math.tau))
        s = h / 16.0
        e.scale = (s * rnd.uniform(0.9, 1.1), s * rnd.uniform(0.9, 1.1), s)
        C["Vegetation"].objects.link(e)

    # --- grass (Geometry Nodes on a helper object that reads the terrain)
    gobj = mesh_obj("Grass_Scatter", [], [], C["Vegetation"], smooth=False)
    mod = gobj.modifiers.new("Grass", "NODES")
    mod.node_group = gn_grass_group()
    set_mod_inputs(mod, {"Ground": terrain, "Tufts": tufts, "Wild": wild, "Density": GRASS_DENSITY})

    # --- birds (7): bodies + flapping wings driven by expressions, keyed flight paths
    bird_mat = mat_simple("MAT_Bird", lin("#2a221c"), 0.7)
    bv, bf = wf.icosphere(2)
    body_v = [(x * 0.07, y * 0.2, z * 0.06) for x, y, z in bv]
    wing_v = [(0, 0.06, 0), (0, -0.07, 0), (0.42, -0.12, 0.01), (0.55, -0.02, 0.02), (0.3, 0.05, 0.005)]
    wing_f = [(0, 1, 2, 4), (4, 2, 3)]
    flocks = [((-90, 70, 30), (10, 60, 46), (110, 85, 40), -1, 19, 3, 3.5),
              ((120, 110, 50), (0, 120, 58), (-110, 95, 36), 3, 18, 2, 4.0),
              ((-160, 140, 70), (20, 160, 80), (160, 130, 62), 6, 16, 2, 3.0)]
    bi = 0
    for (a, m, c, t0, dur, n, spread) in flocks:
        for _ in range(n):
            off = Vector((rnd.uniform(-spread, spread), rnd.uniform(-spread, spread), rnd.uniform(-spread * 0.4, spread * 0.4)))
            path = catmull3([Vector((mx(a[0]), a[1], a[2])) + off, Vector((mx(m[0]), m[1], m[2])) + off,
                             Vector((mx(c[0]), c[1], c[2])) + off], 60)
            body = mesh_obj(f"Bird_{bi}", body_v, bf, C["Birds"])
            assign(body, bird_mat)
            body.scale = (1.6, 1.6, 1.6)
            ph, fr = rnd.uniform(0, 50), rnd.uniform(0.62, 0.8)
            for sgn in (1, -1):
                wing = mesh_obj(f"Bird_{bi}_Wing{'L' if sgn > 0 else 'R'}", [(x * sgn, y, z) for x, y, z in wing_v], wing_f, C["Birds"])
                assign(wing, bird_mat)
                wing.parent = body
                fc = wing.driver_add("rotation_euler", 1)
                fc.driver.expression = f"{sgn} * sin((frame + {ph:.1f}) * {fr:.3f}) * 0.6"
            t0b, durb = t0 + rnd.uniform(-0.6, 0.6), dur * rnd.uniform(0.95, 1.05)
            for fr_ in range(1, FRAMES + 1, 2):
                u = min(max(((fr_ - 1) / FPS - t0b) / durb, 0.0), 1.0)
                k = min(int(u * (len(path) - 1)), len(path) - 2)
                p, q = path[k], path[k + 1]
                body.location = p
                vel = q - p
                body.rotation_euler = (0.0, 0.0, math.atan2(vel.y, vel.x) - math.pi / 2)
                body.keyframe_insert("location", frame=fr_)
                body.keyframe_insert("rotation_euler", frame=fr_)
            bi += 1

    # --- camera: eased glide forward + rise, DOF on the mid-ground
    cam = bpy.data.objects["CAM_Main"]
    cam.data.sensor_width = 36.0
    cam.data.lens = wf.CAM_LENS
    cam.data.dof.use_dof = True
    cam.data.dof.aperture_fstop = FSTOP
    cam.data.dof.aperture_blades = 7
    pos_path = catmull3([(0, -2, 1.8), (mx(0.8), 10, 3.2), (mx(3.5), 24, 6.2), (mx(6.5), 34, 9.5), (mx(8), 40, 12)], 60)
    tgt_path = catmull3([(0, 100, 12.3), (mx(4), 100, 12.5), (mx(12), 100, 13.5), (mx(18), 100, 14), (mx(20), 100, 13)], 60)

    def sample(path, u):
        x = u * (len(path) - 1)
        k = min(int(x), len(path) - 2)
        return path[k].lerp(path[k + 1], x - k)

    prev = None
    for fr_ in range(1, FRAMES + 1):
        t = (fr_ - 1) / (FRAMES - 1)
        u = t * t * t * (t * (t * 6 - 15) + 10)
        p, q = sample(pos_path, u), sample(tgt_path, u)
        cam.location = p
        prev = (q - p).to_track_quat("-Z", "Y").to_euler("XYZ", prev) if prev else (q - p).to_track_quat("-Z", "Y").to_euler()
        cam.rotation_euler = prev
        cam.data.dof.focus_distance = 22.0 + 14.0 * u
        cam.keyframe_insert("location", frame=fr_)
        cam.keyframe_insert("rotation_euler", frame=fr_)
        cam.data.dof.keyframe_insert("focus_distance", frame=fr_)

    # --- hide the asset library from direct rendering
    vl.layer_collection.children["_Assets"].exclude = True
    setup_render(scene, final=True)
    setup_compositor(scene)
    return scene


def setup_render(scene, final=True):
    r = scene.render
    r.engine = "CYCLES"
    r.resolution_x, r.resolution_y, r.resolution_percentage = 1920, 1080, 100
    r.fps = FPS
    scene.frame_start, scene.frame_end = 1, FRAMES
    cy = scene.cycles
    cy.samples = SAMPLES_FINAL if final else SAMPLES_PREVIEW
    cy.use_adaptive_sampling = True
    cy.adaptive_threshold = 0.02
    cy.use_denoising = True
    try:
        cy.denoiser = "OPENIMAGEDENOISE"
    except TypeError:
        pass
    for k, v in (("max_bounces", 8), ("diffuse_bounces", 3), ("glossy_bounces", 3), ("transmission_bounces", 8),
                 ("volume_bounces", 1), ("transparent_max_bounces", 24), ("caustics_reflective", False),
                 ("caustics_refractive", False), ("blur_glossy", 1.0), ("volume_step_rate", 2.0), ("volume_max_steps", 256)):
        try:
            setattr(cy, k, v)
        except Exception:
            pass
    r.use_motion_blur = MOTION_BLUR
    r.motion_blur_shutter = 0.5
    r.use_persistent_data = True
    r.image_settings.file_format = "PNG"
    r.image_settings.color_mode = "RGB"
    r.image_settings.compression = 15
    vs = scene.view_settings
    vs.view_transform = "AgX"
    for look in ("AgX - Medium High Contrast", "Medium High Contrast", "None"):
        try:
            vs.look = look
            break
        except TypeError:
            continue
    vs.exposure = EXPOSURE


def setup_compositor(scene):
    """Glare (fog glow) + subtle chromatic aberration. Grain, vignette and grade come from ffmpeg."""
    try:
        scene.use_nodes = True
        nt = scene.node_tree
        nt.nodes.clear()
        b = NB(nt)
        rl = b.n("CompositorNodeRLayers")
        gl = b.n("CompositorNodeGlare")
        for k, v in (("glare_type", "FOG_GLOW"), ("quality", "HIGH"), ("threshold", 1.2), ("size", 7), ("mix", -0.8)):
            try:
                setattr(gl, k, v)
            except Exception:
                pass
        ld = b.n("CompositorNodeLensdist")
        try:
            ld.use_fit = True
        except Exception:
            pass
        b.setin(ld, ("Dispersion",), 0.01)
        comp = b.n("CompositorNodeComposite")
        b.link(rl.outputs["Image"], gl.inputs["Image"])
        b.link(gl.outputs["Image"], ld.inputs["Image"])
        b.link(ld.outputs["Image"], comp.inputs["Image"])
    except Exception as ex:
        print("WARN compositor skipped:", ex)


def enable_gpu(scene):
    try:
        prefs = bpy.context.preferences.addons["cycles"].preferences
        for dev in ("OPTIX", "CUDA", "HIP", "ONEAPI", "METAL"):
            try:
                prefs.compute_device_type = dev
            except TypeError:
                continue
            for fn in ("refresh_devices", "get_devices"):
                if hasattr(prefs, fn):
                    getattr(prefs, fn)()
                    break
            gpus = [d for d in prefs.devices if d.type == dev]
            if gpus:
                for d in prefs.devices:
                    d.use = d.type == dev
                scene.cycles.device = "GPU"
                print("GPU:", dev, [d.name for d in gpus])
                return dev
    except Exception as ex:
        print("WARN gpu:", ex)
    print("GPU not found - rendering on CPU")
    return "CPU"


def main():
    scene = build()
    save = arg("--save")
    if save:
        bpy.ops.wm.save_as_mainfile(filepath=os.path.abspath(save))
        print("saved", save)
    still = arg("--still", None, int)
    if still:
        enable_gpu(scene)
        if arg("--preview"):
            scene.render.resolution_percentage = 50
            scene.cycles.samples = SAMPLES_PREVIEW
        scene.frame_set(still)
        scene.render.filepath = os.path.abspath(arg("--out", f"still_{still:04d}.png"))
        bpy.ops.render.render(write_still=True)
        print("rendered", scene.render.filepath)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/wf/gpu.py
"""Run before -a: enable the GPU (OptiX/CUDA) and make the frame render resumable."""
import bpy

scene = bpy.context.scene
prefs = bpy.context.preferences.addons["cycles"].preferences
for dev in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dev
    except TypeError:
        continue
    (prefs.refresh_devices if hasattr(prefs, "refresh_devices") else prefs.get_devices)()
    if any(d.type == dev for d in prefs.devices):
        for d in prefs.devices:
            d.use = d.type == dev
        scene.cycles.device = "GPU"
        print("GPU:", dev)
        break
scene.render.use_overwrite = False      # skip frames that already exist (resume after a disconnect)
scene.render.use_placeholder = True


In [ ]:
%%writefile /content/wf/make_audio.sh
#!/bin/bash
# 20 s ambient bed: waterfall rumble + spray, river, wind in leaves, birds, soft pad. Output: ambience.wav
D=20
ffmpeg -y -loglevel error \
 -f lavfi -i "anoisesrc=d=$D:c=brown:a=0.9:seed=11" \
 -f lavfi -i "anoisesrc=d=$D:c=pink:a=0.5:seed=12" \
 -f lavfi -i "anoisesrc=d=$D:c=white:a=0.35:seed=13" \
 -f lavfi -i "aevalsrc='0.10*sin(2*PI*3600*t+40*sin(2*PI*30*t))*between(mod(t,2.3),2.0,2.12)*pow(sin(PI*(mod(t,2.3)-2.0)/0.12),2)+0.08*sin(2*PI*4700*t+60*sin(2*PI*22*t))*between(mod(t+0.9,3.7),3.0,3.09)*pow(sin(PI*(mod(t+0.9,3.7)-3.0)/0.09),2)+0.07*sin(2*PI*2900*t+80*sin(2*PI*14*t))*between(mod(t+1.7,5.3),5.0,5.25)*pow(sin(PI*(mod(t+1.7,5.3)-5.0)/0.25),2)':s=48000:d=$D" \
 -f lavfi -i "aevalsrc='(1-clip((t-9)/2,0,1))*(sin(2*PI*110*t)+0.8*sin(2*PI*164.81*t)+0.6*sin(2*PI*220*t)+0.45*sin(2*PI*277.18*t)+0.35*sin(2*PI*329.63*t))*0.035+clip((t-9)/2,0,1)*(sin(2*PI*92.5*t)+0.8*sin(2*PI*138.59*t)+0.6*sin(2*PI*185*t)+0.45*sin(2*PI*220*t)+0.35*sin(2*PI*277.18*t))*0.035':s=48000:d=$D" \
 -filter_complex "\
[0:a]lowpass=f=700,volume=1.4[fall];\
[1:a]asplit[pk1][pk2];[pk1]highpass=f=1800,lowpass=f=7000,volume=0.25[spray];\
[pk2]bandpass=f=900:width_type=h:w=1400,tremolo=f=0.35:d=0.35,volume=0.5[river];\
[2:a]bandpass=f=3200:width_type=h:w=3000,tremolo=f=0.12:d=0.8,volume=0.18[wind];\
[3:a]aecho=0.8:0.6:40|90:0.25|0.15,volume=0.9[birds];\
[4:a]lowpass=f=1200,tremolo=f=0.1:d=0.4,aecho=0.8:0.7:120|240:0.35|0.2,volume=1.0[pad];\
[fall][spray][river][wind][birds][pad]amix=inputs=6:normalize=0,afade=t=in:d=1.5,afade=t=out:st=17.8:d=2.2,alimiter=limit=0.9,aformat=channel_layouts=stereo[out]" \
 -map "[out]" -ar 48000 ambience.wav


In [ ]:
%%writefile /content/wf/grade_filter.txt
[0:v]format=gbrp,split[base][hi];
[hi]curves=all='0/0 0.8/0 1/1',gblur=sigma=28[glow];
[base][glow]blend=all_mode=screen:all_opacity=0.18,format=yuv444p,
eq=contrast=1.06:saturation=1.16:gamma=0.98,
colorbalance=rs=-0.02:gs=0.01:bs=0.03:rh=0.045:gh=0.025:bh=-0.03,
curves=master='0/0.02 0.25/0.23 0.5/0.51 0.78/0.81 1/0.98',
vignette=angle=PI/7,
noise=alls=3:allf=t,format=yuv420p[v]


In [ ]:
# 5) Build the scene and save it to Drive (about 3-6 minutes, CPU)
!cd /content/wf && /content/blender/blender -b -P final_scene.py -- --save {OUT}/waterfall_final.blend 2>&1 | grep -v -E "^(Read|Fra:)" | tail -n 25

In [ ]:
# 6) PREVIEW: frame 1 at 960x540 with low samples (a few minutes)
!cd /content/wf && /content/blender/blender -b {OUT}/waterfall_final.blend -P gpu.py --python-expr "import bpy; s=bpy.context.scene; s.render.resolution_percentage=50; s.cycles.samples=24; s.render.use_overwrite=True" -o {OUT}/preview_#### -F PNG -f 1 2>&1 | grep -E "GPU|Saved|Error|error" | tail -n 8
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(OUT + '/preview_*.png'))[-1:]:
    display(Image(f, width=960))

In [ ]:
# 7) FULL RENDER: 480 frames, 1920x1080 (resumable - just run again after a disconnect)
!cd /content/wf && /content/blender/blender -b {OUT}/waterfall_final.blend -P gpu.py -o {OUT}/frames/f_#### -F PNG -s 1 -e 480 -a 2>&1 | grep --line-buffered -E "Saved|Error|GPU"
!ls {OUT}/frames | wc -l

In [ ]:
# 8) Colour grade + film grain/vignette + ambient sound -> MP4, then download
!cd /content/wf && bash make_audio.sh && cp ambience.wav {OUT}/
F = open('/content/wf/grade_filter.txt').read().replace('\n', '')
!ffmpeg -y -loglevel error -framerate 24 -i {OUT}/frames/f_%04d.png -i {OUT}/ambience.wav -filter_complex "{F}" -map "[v]" -map 1:a -c:v libx264 -preset slow -crf 17 -pix_fmt yuv420p -c:a aac -b:a 192k -shortest -movflags +faststart {OUT}/waterfall_valley_cycles.mp4
!ls -lh {OUT}/waterfall_valley_cycles.mp4
from google.colab import files
files.download(OUT + '/waterfall_valley_cycles.mp4')